In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

file_path = base_dir / "global_SE_EC2015.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(file_path)

print("Columns:")
print(df.columns.tolist())
print("\nShape before cleaning:", df.shape)

# =========================
# 2. keep 2015 only
# =========================
df["time"] = df["time"].astype(str)
df["year"] = df["time"].str[:4]
df2015 = df[df["year"] == "2015"].copy()

print("\nShape after keeping 2015 only:", df2015.shape)

# =========================
# 3. keep one record per GDLcode
#    sort by time and keep first
# =========================
df2015 = (
    df2015
    .sort_values(["GDLcode", "time"])
    .drop_duplicates(subset=["GDLcode"], keep="first")
    .copy()
)

print("Shape after dropping duplicate GDLcode within 2015:", df2015.shape)

# =========================
# 4. convert key columns to numeric
# =========================
numeric_cols = [
    "population",
    "EC2015_sum",
    "EC2015_mean",
    "EC2015_pixel_count",
    "EC2015_pc",
    "GDP",
    "age",
    "education",
    "gender",
    "Hhsize",
    "income",
    "rural",
    "wealth",
    "HDD_days",
    "total_HDD",
    "mean_temp",
    "threshold"
]

for col in numeric_cols:
    if col in df2015.columns:
        df2015[col] = pd.to_numeric(df2015[col], errors="coerce")

# =========================
# 5. invalid income and wealth to missing
# =========================
for col in ["income", "wealth"]:
    if col in df2015.columns:
        df2015.loc[df2015[col] == -1, col] = np.nan

# =========================
# 6. drop invalid observations
# =========================
before_drop = df2015.shape[0]

df2015 = df2015[
    (df2015["population"] > 0) &
    (df2015["EC2015_sum"] > 0)
].copy()

after_drop = df2015.shape[0]

print("\nDropped rows because of invalid population or EC2015_sum:", before_drop - after_drop)
print("Shape after dropping invalid population and EC2015_sum:", df2015.shape)

# =========================
# 7. recompute per capita electricity consumption
# =========================
df2015["EC2015_pc"] = df2015["EC2015_sum"] / df2015["population"]

# =========================
# 8. build unified SES proxy
#    use income first, otherwise wealth
# =========================
df2015["SES_raw"] = df2015["income"]
mask_missing_ses = df2015["SES_raw"].isna()
df2015.loc[mask_missing_ses, "SES_raw"] = df2015.loc[mask_missing_ses, "wealth"]

# =========================
# 9. missing summary
# =========================
check_vars = [
    "population",
    "EC2015_sum",
    "EC2015_pc",
    "GDP",
    "age",
    "education",
    "gender",
    "Hhsize",
    "income",
    "wealth",
    "SES_raw",
    "rural",
    "country",
    "region",
    "iso_code"
]

check_vars = [c for c in check_vars if c in df2015.columns]

missing_summary = (
    df2015[check_vars]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values summary:")
print(missing_summary)

# =========================
# 10. region summary
# =========================
region_summary = (
    df2015.groupby("region", dropna=False)
    .agg(
        n_gdl=("GDLcode", "count"),
        total_population=("population", "sum"),
        mean_ecpc=("EC2015_pc", "mean"),
        median_ecpc=("EC2015_pc", "median"),
        countries=("iso_code", "nunique"),
        ses_nonmissing=("SES_raw", lambda x: x.notna().sum())
    )
    .sort_values("n_gdl", ascending=False)
)

print("\nRegion summary:")
print(region_summary)

# =========================
# 11. country summary
# =========================
country_summary = (
    df2015.groupby(["region", "country", "iso_code"], dropna=False)
    .agg(
        n_gdl=("GDLcode", "count"),
        total_population=("population", "sum"),
        mean_ecpc=("EC2015_pc", "mean"),
        median_ecpc=("EC2015_pc", "median"),
        ses_nonmissing=("SES_raw", lambda x: x.notna().sum())
    )
    .reset_index()
    .sort_values(["region", "n_gdl"], ascending=[True, False])
)

print("\nTop 20 countries by number of GDL units:")
print(country_summary.head(20))

# =========================
# 12. quick consistency check within one year
#    see whether monthly records in 2015 repeat
# =========================
raw_2015 = df[df["year"] == "2015"].copy()

sample_gdl = raw_2015["GDLcode"].dropna().astype(str).iloc[0]
sample_check = raw_2015.loc[raw_2015["GDLcode"].astype(str) == sample_gdl, [
    c for c in [
        "GDLcode", "time", "GDP", "population", "age", "education",
        "gender", "Hhsize", "income", "rural", "wealth",
        "HDD_days", "total_HDD", "mean_temp", "EC2015_sum", "EC2015_pc"
    ] if c in raw_2015.columns
]].copy()

print("\nSample within 2015 for one GDLcode:")
print(sample_check.head(12))

# =========================
# 13. save intermediate files
# =========================
clean_base_path = process_dir / "df2015_clean_base.csv"
missing_path = process_dir / "missing_summary_2015.csv"
region_path = process_dir / "region_summary_2015.csv"
country_path = process_dir / "country_summary_2015.csv"
sample_check_path = process_dir / "sample_gdl_within_2015_check.csv"

df2015.to_csv(clean_base_path, index=False)
missing_summary.to_csv(missing_path, header=["missing_count"])
region_summary.to_csv(region_path)
country_summary.to_csv(country_path, index=False)
sample_check.to_csv(sample_check_path, index=False)

print("\nSaved files:")
print(clean_base_path)
print(missing_path)
print(region_path)
print(country_path)
print(sample_check_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_clean_base.csv"

# =========================
# 1. read cleaned base table
# =========================
df = pd.read_csv(input_path)

print("Input shape:", df.shape)

# =========================
# 2. drop rows with missing key identifiers
# =========================
key_id_cols = ["GDLcode", "iso_code", "country"]
for col in key_id_cols:
    df[col] = df[col].replace("", np.nan)

before_drop = len(df)
df = df.dropna(subset=key_id_cols).copy()
after_drop = len(df)

print("Dropped rows with missing key identifiers:", before_drop - after_drop)
print("Shape after dropping missing identifiers:", df.shape)

# =========================
# 3. build 7 analytical regions from iso_code
# =========================
sub_saharan_africa = {
    "AGO","BDI","BEN","BFA","BWA","CAF","CIV","CMR","COD","COG","COM","CPV","DJI","ERI",
    "ETH","GAB","GHA","GIN","GMB","GNB","GNQ","KEN","LBR","LSO","MDG","MLI","MOZ","MRT",
    "MUS","MWI","NAM","NER","NGA","RWA","SDN","SEN","SLE","SOM","SSD","STP","SWZ","SYC",
    "TCD","TGO","TZA","UGA","ZAF","ZMB","ZWE"
}

north_africa_western_asia = {
    "AFG","ARE","ARM","AZE","BHR","DZA","EGY","GEO","IRN","IRQ","ISR","JOR","KAZ","KGZ",
    "KWT","LBN","LBY","MAR","OMN","PSE","QAT","SAU","SYR","TJK","TKM","TUN","TUR","UZB","YEM"
}

south_asia = {
    "BGD","BTN","IND","LKA","MDV","NPL","PAK"
}

east_southeast_asia = {
    "BRN","CHN","IDN","JPN","KHM","KOR","LAO","MMR","MNG","MYS","PHL","PRK","SGP","THA","TLS","VNM"
}

europe = {
    "ALB","AND","AUT","BEL","BGR","BIH","BLR","CHE","CYP","CZE","DEU","DNK","ESP","EST","FIN",
    "FRA","GBR","GRC","HRV","HUN","IRL","ISL","ITA","LTU","LUX","LVA","MDA","MKD","MLT","MNE",
    "NLD","NOR","POL","PRT","ROU","RUS","SRB","SVK","SVN","SWE","UKR"
}

latin_america_caribbean = {
    "ARG","ATG","BHS","BLZ","BOL","BRA","BRB","CHL","COL","CRI","CUB","DMA","DOM","ECU","GRD",
    "GTM","GUY","HND","HTI","JAM","KNA","LCA","MEX","NIC","PAN","PER","PRY","SLV","SUR","TTO",
    "URY","VCT","VEN"
}

north_america_oceania = {
    "AUS","CAN","FSM","KIR","MHL","NRU","NZL","PLW","PNG","SLB","TON","TUV","USA","VUT","WSM"
}

def map_analytical_region(iso):
    if pd.isna(iso):
        return np.nan
    iso = str(iso).strip().upper()
    if iso in sub_saharan_africa:
        return "Sub-Saharan Africa"
    elif iso in north_africa_western_asia:
        return "Northern Africa and Western Asia"
    elif iso in south_asia:
        return "South Asia"
    elif iso in east_southeast_asia:
        return "Eastern and South-eastern Asia"
    elif iso in europe:
        return "Europe"
    elif iso in latin_america_caribbean:
        return "Latin America and the Caribbean"
    elif iso in north_america_oceania:
        return "Northern America and Oceania"
    else:
        return np.nan

df["analytical_region"] = df["iso_code"].apply(map_analytical_region)

print("\nAnalytical region value counts:")
print(df["analytical_region"].value_counts(dropna=False))

# =========================
# 4. drop rows not assigned to any analytical region
# =========================
before_region_drop = len(df)
df = df.dropna(subset=["analytical_region"]).copy()
after_region_drop = len(df)

print("\nDropped rows without analytical region:", before_region_drop - after_region_drop)
print("Shape after analytical region assignment:", df.shape)

# =========================
# 5. make sure target variables are numeric
# =========================
vars_for_grouping = ["SES_raw", "rural", "education", "age", "gender", "Hhsize"]

for col in vars_for_grouping + ["population", "EC2015_pc"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# =========================
# 6. helper function for country internal terciles
# =========================
def make_country_tercile_group(data, value_col, country_col="iso_code"):
    """
    Return a categorical tercile group within each country.
    Output labels:
    1 = low
    2 = middle
    3 = high

    Rules:
    1. If missing, remain missing
    2. If a country has fewer than 3 nonmissing observations, return missing
    3. If too many ties make qcut fail, use rank based qcut
    """
    out = pd.Series(index=data.index, dtype="float")

    for iso, sub in data.groupby(country_col):
        x = sub[value_col]
        valid = x.notna()

        if valid.sum() < 3:
            out.loc[sub.index] = np.nan
            continue

        x_valid = x.loc[valid]

        try:
            grp = pd.qcut(x_valid, q=3, labels=[1, 2, 3], duplicates="drop")
            grp = grp.astype(float)

            # if duplicates drop reduced bins, force missing for this country
            if grp.nunique() < 3:
                ranks = x_valid.rank(method="average")
                grp = pd.qcut(ranks, q=3, labels=[1, 2, 3], duplicates="drop").astype(float)

            if grp.nunique() < 3:
                out.loc[sub.index] = np.nan
            else:
                out.loc[x_valid.index] = grp

        except Exception:
            try:
                ranks = x_valid.rank(method="average")
                grp = pd.qcut(ranks, q=3, labels=[1, 2, 3], duplicates="drop").astype(float)

                if grp.nunique() < 3:
                    out.loc[sub.index] = np.nan
                else:
                    out.loc[x_valid.index] = grp
            except Exception:
                out.loc[sub.index] = np.nan

    return out

# =========================
# 7. build tercile groups
# =========================
group_var_map = {
    "SES_raw": "ses_group",
    "rural": "rural_group",
    "education": "education_group",
    "age": "age_group",
    "gender": "gender_group",
    "Hhsize": "hhsize_group"
}

for raw_col, group_col in group_var_map.items():
    df[group_col] = make_country_tercile_group(df, raw_col)

label_map = {1.0: "low", 2.0: "middle", 3.0: "high"}

for group_col in group_var_map.values():
    label_col = f"{group_col}_label"
    df[label_col] = df[group_col].map(label_map)

# =========================
# 8. diagnostics for grouping success
# =========================
grouping_diagnostics = []

for raw_col, group_col in group_var_map.items():
    temp = (
        df.groupby("iso_code")
        .agg(
            country=("country", "first"),
            nonmissing_raw=(raw_col, lambda x: x.notna().sum()),
            nonmissing_group=(group_col, lambda x: x.notna().sum()),
            n_groups=(group_col, lambda x: pd.Series(x.dropna()).nunique())
        )
        .reset_index()
    )
    temp["attribute"] = raw_col
    grouping_diagnostics.append(temp)

grouping_diagnostics = pd.concat(grouping_diagnostics, ignore_index=True)

print("\nGrouping diagnostics, first 30 rows:")
print(grouping_diagnostics.head(30))

# =========================
# 9. analytical region summary based on corrected regions
# =========================
analytical_region_summary = (
    df.groupby("analytical_region")
    .agg(
        n_gdl=("GDLcode", "count"),
        countries=("iso_code", "nunique"),
        total_population=("population", "sum"),
        mean_ecpc=("EC2015_pc", "mean"),
        median_ecpc=("EC2015_pc", "median"),
        ses_group_nonmissing=("ses_group", lambda x: x.notna().sum()),
        rural_group_nonmissing=("rural_group", lambda x: x.notna().sum()),
        age_group_nonmissing=("age_group", lambda x: x.notna().sum()),
        gender_group_nonmissing=("gender_group", lambda x: x.notna().sum())
    )
    .sort_values("n_gdl", ascending=False)
)

print("\nAnalytical region summary:")
print(analytical_region_summary)

# =========================
# 10. save process files
# =========================
analytical_path = process_dir / "df2015_analytical.csv"
grouping_diag_path = process_dir / "grouping_diagnostics.csv"
analytical_region_summary_path = process_dir / "analytical_region_summary_2015.csv"

df.to_csv(analytical_path, index=False)
grouping_diagnostics.to_csv(grouping_diag_path, index=False)
analytical_region_summary.to_csv(analytical_region_summary_path)

print("\nSaved files:")
print(analytical_path)
print(grouping_diag_path)
print(analytical_region_summary_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"

# =========================
# 1. read analytical table
# =========================
df = pd.read_csv(input_path)

print("Input shape:", df.shape)

# make sure numeric
for col in ["population", "EC2015_pc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep valid rows for overall inequality
df_overall = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] >= 0) &
    df["analytical_region"].notna()
].copy()

print("Shape used for overall inequality:", df_overall.shape)

# =========================
# 2. weighted gini function
# =========================
def weighted_gini(x, w):
    """
    Population weighted Gini coefficient.
    x: outcome array
    w: weight array
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan

    if np.all(x == 0):
        return 0.0

    order = np.argsort(x)
    x = x[order]
    w = w[order]

    wx = w * x
    W = w.sum()
    Y = wx.sum()

    if W <= 0 or Y <= 0:
        return np.nan

    p = np.cumsum(w) / W
    L = np.cumsum(wx) / Y

    p_prev = np.concatenate(([0], p[:-1]))
    L_prev = np.concatenate(([0], L[:-1]))

    area = np.sum((L + L_prev) * (p - p_prev) / 2)
    gini = 1 - 2 * area

    return gini

# =========================
# 3. Lorenz points function
# =========================
def lorenz_points(x, w):
    """
    Return Lorenz curve points for weighted data.
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return pd.DataFrame(columns=["cum_pop_share", "cum_ec_share"])

    order = np.argsort(x)
    x = x[order]
    w = w[order]

    wx = w * x
    W = w.sum()
    Y = wx.sum()

    if Y <= 0:
        out = pd.DataFrame({
            "cum_pop_share": np.concatenate(([0], np.cumsum(w) / W)),
            "cum_ec_share": np.zeros(len(w) + 1)
        })
        return out

    out = pd.DataFrame({
        "cum_pop_share": np.concatenate(([0], np.cumsum(w) / W)),
        "cum_ec_share": np.concatenate(([0], np.cumsum(wx) / Y))
    })
    return out

# =========================
# 4. region level Gini
# =========================
region_records = []
lorenz_records = []

region_list = sorted(df_overall["analytical_region"].dropna().unique())

for region in tqdm(region_list, desc="Computing region Gini and Lorenz"):
    sub = df_overall[df_overall["analytical_region"] == region].copy()

    gini_val = weighted_gini(sub["EC2015_pc"], sub["population"])

    region_records.append({
        "analytical_region": region,
        "n_gdl": len(sub),
        "n_countries": sub["iso_code"].nunique(),
        "total_population": sub["population"].sum(),
        "mean_ecpc": np.average(sub["EC2015_pc"], weights=sub["population"]),
        "median_ecpc_unweighted": sub["EC2015_pc"].median(),
        "gini": gini_val
    })

    lor = lorenz_points(sub["EC2015_pc"], sub["population"]).copy()
    lor["analytical_region"] = region
    lor["point_id"] = np.arange(len(lor))
    lorenz_records.append(lor)

region_gini = pd.DataFrame(region_records).sort_values("gini", ascending=False)
region_lorenz_points = pd.concat(lorenz_records, ignore_index=True)

print("\nRegion Gini summary:")
print(region_gini)

# =========================
# 5. country level Gini
#    keep this for later Figure 3
# =========================
country_records = []

country_groups = list(df_overall.groupby(["iso_code", "country", "analytical_region"]))

for (iso, country, region), sub in tqdm(country_groups, desc="Computing country Gini"):
    if len(sub) < 2:
        continue

    gini_val = weighted_gini(sub["EC2015_pc"], sub["population"])

    country_records.append({
        "iso_code": iso,
        "country": country,
        "analytical_region": region,
        "n_gdl": len(sub),
        "total_population": sub["population"].sum(),
        "mean_ecpc": np.average(sub["EC2015_pc"], weights=sub["population"]),
        "median_ecpc_unweighted": sub["EC2015_pc"].median(),
        "gini": gini_val
    })

country_gini = pd.DataFrame(country_records).sort_values(
    ["analytical_region", "gini"], ascending=[True, False]
)

print("\nCountry Gini preview:")
print(country_gini.head(20))

# =========================
# 6. save process files
# =========================
region_gini_path = process_dir / "region_gini_2015.csv"
region_lorenz_path = process_dir / "region_lorenz_points_2015.csv"
country_gini_path = process_dir / "country_gini_2015.csv"

region_gini.to_csv(region_gini_path, index=False)
region_lorenz_points.to_csv(region_lorenz_path, index=False)
country_gini.to_csv(country_gini_path, index=False)

print("\nSaved files:")
print(region_gini_path)
print(region_lorenz_path)
print(country_gini_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

# 注意：这里默认 geopandas 已安装
import geopandas as gpd

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

region_gini_path = process_dir / "region_gini_2015.csv"
region_lorenz_path = process_dir / "region_lorenz_points_2015.csv"
analytical_path = process_dir / "df2015_analytical.csv"

shp_path = base_dir / "Data" / "GDL Shapefiles V4" / "GDL Shapefiles V4.shp"

# =========================
# 1. read data
# =========================
region_gini = pd.read_csv(region_gini_path)
region_lorenz = pd.read_csv(region_lorenz_path)
df = pd.read_csv(analytical_path)

print("region_gini shape:", region_gini.shape)
print("region_lorenz shape:", region_lorenz.shape)
print("analytical table shape:", df.shape)

# =========================
# 2. sort Figure 1 by Gini high to low
# =========================
region_gini = region_gini.sort_values("gini", ascending=False).reset_index(drop=True)
region_order = region_gini["analytical_region"].tolist()

region_lorenz["analytical_region"] = pd.Categorical(
    region_lorenz["analytical_region"],
    categories=region_order,
    ordered=True
)
region_lorenz = region_lorenz.sort_values(["analytical_region", "point_id"]).reset_index(drop=True)

print("\nRegion order by Gini, high to low:")
print(region_gini[["analytical_region", "gini"]])

# =========================
# 3. labels and colors
# =========================
region_label_map = {
    "Sub-Saharan Africa": "Sub-Saharan Africa",
    "Northern Africa and Western Asia": "Northern Africa and Western Asia",
    "Eastern and South-eastern Asia": "Eastern and South-eastern Asia",
    "Latin America and the Caribbean": "Latin America and the Caribbean",
    "South Asia": "South Asia",
    "Europe": "Europe",
    "Northern America and Oceania": "Northern America and Oceania"
}

palette = {
    "Sub-Saharan Africa": "#8c2d2d",
    "Northern Africa and Western Asia": "#c76d3a",
    "Eastern and South-eastern Asia": "#2f6c8f",
    "Latin America and the Caribbean": "#7a5aa6",
    "South Asia": "#d8a31a",
    "Europe": "#4c8c4a",
    "Northern America and Oceania": "#c05a8a"
}

# =========================
# 4. Figure 1, 4 plus 3 layout
# =========================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.9
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12

fig = plt.figure(figsize=(20, 10.5))
gs = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.22)

axes = []
for j in range(4):
    axes.append(fig.add_subplot(gs[0, j]))
for j in range(3):
    axes.append(fig.add_subplot(gs[1, j]))

# last blank panel in bottom right
blank_ax = fig.add_subplot(gs[1, 3])
blank_ax.axis("off")

for i, region in enumerate(region_order):
    ax = axes[i]

    sub_curve = region_lorenz.loc[region_lorenz["analytical_region"] == region].copy()
    sub_info = region_gini.loc[region_gini["analytical_region"] == region].copy()

    color = palette.get(region, "#4C72B0")
    gini_val = sub_info["gini"].iloc[0]

    x = sub_curve["cum_pop_share"].values
    y = sub_curve["cum_ec_share"].values

    ax.fill_between(x, y, 0, color=color, alpha=0.20, zorder=1)
    ax.plot(
        x, y,
        color=color,
        linewidth=3.6,
        solid_capstyle="round",
        zorder=3
    )
    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        linewidth=1.2,
        color="#9a9a9a",
        zorder=2
    )

    ax.set_title(region_label_map.get(region, region), pad=10, fontsize=16)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xlabel("Cumulative population share", fontsize=14)
    ax.set_ylabel("Cumulative electricity share", fontsize=14)

    ax.text(
        0.05, 0.90,
        f"Gini = {gini_val:.3f}",
        transform=ax.transAxes,
        fontsize=14,
        ha="left",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor="white",
            edgecolor="none",
            alpha=0.92
        )
    )

    ax.grid(True, alpha=0.16, linewidth=0.6)

fig.suptitle(
    "Figure 1. Region level overall inequality in per capita electricity consumption",
    fontsize=20,
    y=0.98
)

fig1_png = results_dir / "Figure1_region_lorenz_curves_sorted_4plus3.png"
fig1_pdf = results_dir / "Figure1_region_lorenz_curves_sorted_4plus3.pdf"

plt.savefig(fig1_png, dpi=500, bbox_inches="tight")
plt.savefig(fig1_pdf, bbox_inches="tight")
plt.show()

print("\nSaved Figure 1:")
print(fig1_png)
print(fig1_pdf)

# =========================
# 5. prepare global map data
# =========================
# keep only columns needed for mapping
map_df = df[[
    "GDLcode", "iso_code", "country", "analytical_region", "EC2015_pc", "population"
]].copy()

map_df["EC2015_pc"] = pd.to_numeric(map_df["EC2015_pc"], errors="coerce")
map_df["population"] = pd.to_numeric(map_df["population"], errors="coerce")
map_df["GDLcode"] = map_df["GDLcode"].astype(str)

print("\nReading shapefile, this may take a little time...")
gdf = gpd.read_file(shp_path)

print("Raw shapefile shape:", gdf.shape)
print("Shapefile columns:", gdf.columns.tolist())

gdf["GDLcode"] = gdf["GDLcode"].astype(str)

# merge analytical data into geometry
gdf_map = gdf.merge(map_df, on="GDLcode", how="left")

print("Merged map shape:", gdf_map.shape)
print("Mapped observations with EC2015_pc:", gdf_map["EC2015_pc"].notna().sum())

# save merged process file
merged_geo_path = process_dir / "gdf_ec2015pc_analytical.geojson"
gdf_map.to_file(merged_geo_path, driver="GeoJSON")
print("Saved merged geodata:", merged_geo_path)

# =========================
# 6. create region boundary layer
# =========================
region_boundary_parts = []

for region in tqdm(sorted(gdf_map["analytical_region"].dropna().unique()), desc="Building region boundaries"):
    sub = gdf_map[gdf_map["analytical_region"] == region].copy()

    if len(sub) == 0:
        continue

    # dissolve to multi polygon region footprint
    dissolved = sub[["analytical_region", "geometry"]].dissolve(by="analytical_region")
    dissolved["analytical_region"] = dissolved.index
    region_boundary_parts.append(dissolved.reset_index(drop=True))

region_boundaries = pd.concat(region_boundary_parts, ignore_index=True)
region_boundaries = gpd.GeoDataFrame(region_boundaries, geometry="geometry", crs=gdf_map.crs)

region_boundary_path = process_dir / "analytical_region_boundaries.geojson"
region_boundaries.to_file(region_boundary_path, driver="GeoJSON")
print("Saved region boundaries:", region_boundary_path)

# =========================
# 7. determine color scale for global map
# =========================
valid_vals = gdf_map["EC2015_pc"].replace([np.inf, -np.inf], np.nan).dropna()
valid_vals = valid_vals[valid_vals > 0]

vmin = valid_vals.quantile(0.02)
vmax = valid_vals.quantile(0.98)

print("\nColor scale:")
print("vmin:", vmin)
print("vmax:", vmax)

# =========================
# 8. plot global map
# =========================
fig, ax = plt.subplots(figsize=(22, 10))

# background for all polygons
gdf_map.plot(
    ax=ax,
    color="#e8e8e8",
    edgecolor="white",
    linewidth=0.12
)

# valid filled polygons
gdf_map.loc[gdf_map["EC2015_pc"].notna() & (gdf_map["EC2015_pc"] > 0)].plot(
    ax=ax,
    column="EC2015_pc",
    cmap="YlGnBu",
    norm=LogNorm(vmin=vmin, vmax=vmax),
    linewidth=0.10,
    edgecolor="#f7f7f7",
    legend=True,
    legend_kwds={
        "label": "Per capita electricity consumption",
        "orientation": "horizontal",
        "shrink": 0.42,
        "pad": 0.03
    }
)

# overlay analytical region boundaries
for region in region_order:
    sub = region_boundaries[region_boundaries["analytical_region"] == region]
    if len(sub) == 0:
        continue

    sub.boundary.plot(
        ax=ax,
        color=palette[region],
        linewidth=1.0,
        alpha=0.95
    )

# map appearance
ax.set_title(
    "Global distribution of per capita electricity consumption in 2015",
    fontsize=20,
    pad=16
)
ax.set_xlabel("Longitude", fontsize=15)
ax.set_ylabel("Latitude", fontsize=15)
ax.tick_params(axis="both", labelsize=13)

# standard global extent
ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)

# legend for analytical region boundaries
legend_handles = [
    Patch(facecolor=palette[r], edgecolor=palette[r], label=r)
    for r in region_order
]
leg = ax.legend(
    handles=legend_handles,
    title="Analytical regions",
    loc="lower left",
    bbox_to_anchor=(0.01, 0.02),
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    ncol=2
)

# add panel letter if you later combine figures
ax.text(
    0.01, 0.97, "A",
    transform=ax.transAxes,
    fontsize=28,
    fontweight="bold",
    va="top",
    ha="left"
)

fig.tight_layout()

global_map_png = results_dir / "Global_map_EC2015_pc_with_regions.png"
global_map_pdf = results_dir / "Global_map_EC2015_pc_with_regions.pdf"

plt.savefig(global_map_png, dpi=500, bbox_inches="tight")
plt.savefig(global_map_pdf, bbox_inches="tight")
plt.show()

print("\nSaved global map:")
print(global_map_png)
print(global_map_pdf)

# =========================
# 9. optional summary table for the map
# =========================
map_region_summary = (
    gdf_map.groupby("analytical_region")
    .agg(
        n_gdl=("GDLcode", "count"),
        n_valid_ecpc=("EC2015_pc", lambda x: x.notna().sum()),
        total_population=("population", "sum"),
        mean_ecpc=("EC2015_pc", "mean"),
        median_ecpc=("EC2015_pc", "median")
    )
    .reset_index()
)

map_region_summary_path = process_dir / "global_map_region_summary.csv"
map_region_summary.to_csv(map_region_summary_path, index=False)

print("Saved summary table:")
print(map_region_summary_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, PowerNorm
from matplotlib.cm import ScalarMappable

import geopandas as gpd

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

analytical_csv = process_dir / "df2015_analytical.csv"
shp_path = base_dir / "Data" / "GDL Shapefiles V4" / "GDL Shapefiles V4.shp"

diag_path = process_dir / "top50_highest_ecpc_for_map_diagnosis.csv"
map_summary_path = process_dir / "strict_blank_map_summary.csv"
scheme_summary_path = process_dir / "global_map_scheme_summary.csv"

# =========================
# 1. read analytical table
# =========================
if not analytical_csv.exists():
    raise FileNotFoundError(f"Missing file: {analytical_csv}")

df = pd.read_csv(analytical_csv)

required_cols = ["GDLcode", "iso_code", "country", "analytical_region", "EC2015_pc", "population"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in df2015_analytical.csv: {missing_cols}")

df_map = df[required_cols].copy()
df_map["GDLcode"] = df_map["GDLcode"].astype(str)
df_map["EC2015_pc"] = pd.to_numeric(df_map["EC2015_pc"], errors="coerce")
df_map["population"] = pd.to_numeric(df_map["population"], errors="coerce")

print("Analytical table shape:", df_map.shape)

# =========================
# 2. read shapefile and merge
# =========================
if not shp_path.exists():
    raise FileNotFoundError(f"Missing shapefile: {shp_path}")

print("Reading shapefile ...")
gdf = gpd.read_file(shp_path)
gdf["GDLcode"] = gdf["GDLcode"].astype(str)

print("Shapefile shape:", gdf.shape)
print("Merging shapefile with analytical table ...")

gdf_map = gdf.merge(df_map, on="GDLcode", how="left", suffixes=("_shp", "_ana"))

print("Merged map shape:", gdf_map.shape)
print("Merged columns:")
print(gdf_map.columns.tolist())

# =========================
# 3. harmonize merged columns
# =========================
def pick_col(df_obj, preferred_names):
    for name in preferred_names:
        if name in df_obj.columns:
            return name
    return None

iso_col = pick_col(gdf_map, ["iso_code_ana", "iso_code"])
country_col = pick_col(gdf_map, ["country_ana", "country"])
region_col = pick_col(gdf_map, ["analytical_region"])
ecpc_col = pick_col(gdf_map, ["EC2015_pc_ana", "EC2015_pc"])
pop_col = pick_col(gdf_map, ["population_ana", "population"])

print("\nSelected columns after merge:")
print("iso_col =", iso_col)
print("country_col =", country_col)
print("region_col =", region_col)
print("ecpc_col =", ecpc_col)
print("pop_col =", pop_col)

if any(v is None for v in [iso_col, country_col, region_col, ecpc_col, pop_col]):
    raise KeyError("Failed to locate one or more required columns after merge.")

gdf_map["iso_code_use"] = gdf_map[iso_col]
gdf_map["country_use"] = gdf_map[country_col]
gdf_map["analytical_region_use"] = gdf_map[region_col]
gdf_map["EC2015_pc_use"] = pd.to_numeric(gdf_map[ecpc_col], errors="coerce")
gdf_map["population_use"] = pd.to_numeric(gdf_map[pop_col], errors="coerce")

gdf_map.loc[~np.isfinite(gdf_map["EC2015_pc_use"]), "EC2015_pc_use"] = np.nan
gdf_map.loc[~np.isfinite(gdf_map["population_use"]), "population_use"] = np.nan

# =========================
# 4. build region boundaries on the fly
# =========================
print("\nBuilding analytical region boundaries ...")

boundary_parts = []
valid_regions = sorted(gdf_map["analytical_region_use"].dropna().unique())

for region in tqdm(valid_regions, desc="Building region boundaries"):
    sub = gdf_map.loc[
        gdf_map["analytical_region_use"] == region,
        ["analytical_region_use", "geometry"]
    ].copy()

    if len(sub) == 0:
        continue

    dissolved = sub.dissolve(by="analytical_region_use")
    dissolved["analytical_region"] = dissolved.index
    boundary_parts.append(
        dissolved.reset_index(drop=True)[["analytical_region", "geometry"]]
    )

region_boundaries = pd.concat(boundary_parts, ignore_index=True)
region_boundaries = gpd.GeoDataFrame(region_boundaries, geometry="geometry", crs=gdf_map.crs)

print("region_boundaries shape:", region_boundaries.shape)

# =========================
# 5. high-value diagnosis
# =========================
diagnostic = gdf_map[[
    "GDLcode", "geometry", "iso_code_use", "country_use",
    "analytical_region_use", "population_use", "EC2015_pc_use"
]].copy()

diagnostic = diagnostic.rename(columns={
    "iso_code_use": "iso_code",
    "country_use": "country",
    "analytical_region_use": "analytical_region",
    "population_use": "population",
    "EC2015_pc_use": "EC2015_pc"
})

rep_points = diagnostic.geometry.representative_point()
diagnostic["rep_x"] = rep_points.x
diagnostic["rep_y"] = rep_points.y

suspicious = diagnostic[
    diagnostic["population"].notna() &
    diagnostic["EC2015_pc"].notna()
].copy()

suspicious = suspicious.sort_values("EC2015_pc", ascending=False)
top50 = suspicious.head(50).copy()
top50.drop(columns="geometry").to_csv(diag_path, index=False)

print("\nTop 20 highest EC2015_pc units for diagnosis:")
print(top50.drop(columns="geometry").head(20))

# =========================
# 6. strict blanking for map only
# =========================
gdf_plot = gdf_map.copy()

valid_mask = (
    gdf_plot["population_use"].notna() &
    gdf_plot["EC2015_pc_use"].notna() &
    np.isfinite(gdf_plot["population_use"]) &
    np.isfinite(gdf_plot["EC2015_pc_use"]) &
    (gdf_plot["population_use"] >= 1000) &
    (gdf_plot["EC2015_pc_use"] > 0)
)

gdf_plot["EC2015_pc_map"] = np.nan
gdf_plot.loc[valid_mask, "EC2015_pc_map"] = gdf_plot.loc[valid_mask, "EC2015_pc_use"]

print("\nMap-valid observations:", int(valid_mask.sum()))
print("Blanked observations:", int((~valid_mask).sum()))

valid_vals = gdf_plot["EC2015_pc_map"].dropna()
valid_vals = valid_vals[valid_vals > 0]

if len(valid_vals) == 0:
    raise ValueError("No valid map values left after blanking.")

# =========================
# 7. save summary
# =========================
summary_df = pd.DataFrame({
    "metric": [
        "total_units",
        "valid_map_units",
        "blanked_units",
        "population_threshold"
    ],
    "value": [
        len(gdf_plot),
        int(valid_mask.sum()),
        int((~valid_mask).sum()),
        1000
    ]
})
summary_df.to_csv(map_summary_path, index=False)

# =========================
# 8. region order and stronger boundary colors
# =========================
region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "Eastern and South-eastern Asia",
    "Latin America and the Caribbean",
    "South Asia",
    "Europe",
    "Northern America and Oceania"
]

region_boundary_palette = {
    "Sub-Saharan Africa": "#b2182b",
    "Northern Africa and Western Asia": "#ef8a62",
    "Eastern and South-eastern Asia": "#2166ac",
    "Latin America and the Caribbean": "#762a83",
    "South Asia": "#e6ab02",
    "Europe": "#1b7837",
    "Northern America and Oceania": "#c51b7d"
}

# =========================
# 9. plotting style
# =========================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.9

# =========================
# 10. candidate schemes
# =========================
schemes = [
    {"scheme_id": "S1_cividis_log_q02_q98", "cmap": "cividis", "norm_type": "log", "q_low": 0.02, "q_high": 0.98},
    {"scheme_id": "S2_cividis_log_q05_q95", "cmap": "cividis", "norm_type": "log", "q_low": 0.05, "q_high": 0.95},
    {"scheme_id": "S3_viridis_log_q02_q98", "cmap": "viridis", "norm_type": "log", "q_low": 0.02, "q_high": 0.98},
    {"scheme_id": "S4_viridis_power_q02_q98_g05", "cmap": "viridis", "norm_type": "power", "q_low": 0.02, "q_high": 0.98, "gamma": 0.5},
    {"scheme_id": "S5_magma_log_q02_q98", "cmap": "magma", "norm_type": "log", "q_low": 0.02, "q_high": 0.98},
    {"scheme_id": "S6_magma_power_q02_q98_g04", "cmap": "magma", "norm_type": "power", "q_low": 0.02, "q_high": 0.98, "gamma": 0.4},
    {"scheme_id": "S7_YlGnBu_log_q05_q95", "cmap": "YlGnBu", "norm_type": "log", "q_low": 0.05, "q_high": 0.95},
    {"scheme_id": "S8_YlGnBu_power_q02_q98_g05", "cmap": "YlGnBu", "norm_type": "power", "q_low": 0.02, "q_high": 0.98, "gamma": 0.5},
]

scheme_records = []

# =========================
# 11. generate candidate maps
# =========================
for scheme in tqdm(schemes, desc="Generating map candidates"):
    scheme_id = scheme["scheme_id"]
    cmap = scheme["cmap"]
    q_low = scheme["q_low"]
    q_high = scheme["q_high"]

    vmin = valid_vals.quantile(q_low)
    vmax = valid_vals.quantile(q_high)

    if vmin <= 0:
        vmin = valid_vals.min()
    if vmax <= vmin:
        vmax = valid_vals.max()

    gdf_scheme = gdf_plot.copy()
    gdf_scheme["EC2015_pc_map_clip"] = gdf_scheme["EC2015_pc_map"].clip(lower=vmin, upper=vmax)

    if scheme["norm_type"] == "log":
        norm = LogNorm(vmin=vmin, vmax=vmax)
    elif scheme["norm_type"] == "power":
        gamma = scheme.get("gamma", 0.5)
        norm = PowerNorm(gamma=gamma, vmin=vmin, vmax=vmax)
    else:
        raise ValueError("Unknown norm_type")

    fig, ax = plt.subplots(figsize=(22, 10))

    # background
    gdf_scheme.plot(
        ax=ax,
        color="#efefef",
        edgecolor="white",
        linewidth=0.08
    )

    # valid polygons
    gdf_scheme.loc[gdf_scheme["EC2015_pc_map_clip"].notna()].plot(
        ax=ax,
        column="EC2015_pc_map_clip",
        cmap=cmap,
        norm=norm,
        linewidth=0.05,
        edgecolor="#f5f5f5",
        legend=False
    )

    # region boundaries
    for region in region_order:
        sub = region_boundaries[region_boundaries["analytical_region"] == region]
        if len(sub) == 0:
            continue
        sub.boundary.plot(
            ax=ax,
            color=region_boundary_palette[region],
            linewidth=1.45,
            alpha=1.0
        )

    ax.set_title(
        "Global distribution of per capita electricity consumption in 2015",
        fontsize=22,
        pad=16
    )
    ax.set_xlabel("Longitude", fontsize=16)
    ax.set_ylabel("Latitude", fontsize=16)
    ax.tick_params(axis="both", labelsize=13)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)

    ax.text(
        0.01, 0.97, "A",
        transform=ax.transAxes,
        fontsize=28,
        fontweight="bold",
        va="top",
        ha="left"
    )

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.065,
        shrink=0.48
    )
    cbar.set_label("Per capita electricity consumption", fontsize=16)
    cbar.ax.tick_params(labelsize=13)

    fig.tight_layout()

    out_png = results_dir / f"{scheme_id}.png"
    out_pdf = results_dir / f"{scheme_id}.pdf"

    plt.savefig(out_png, dpi=500, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

    scheme_records.append({
        "scheme_id": scheme_id,
        "cmap": cmap,
        "norm_type": scheme["norm_type"],
        "q_low": q_low,
        "q_high": q_high,
        "gamma": scheme.get("gamma", np.nan),
        "vmin": float(vmin),
        "vmax": float(vmax),
        "png_file": str(out_png),
        "pdf_file": str(out_pdf)
    })

# =========================
# 12. save scheme summary
# =========================
scheme_summary = pd.DataFrame(scheme_records)
scheme_summary.to_csv(scheme_summary_path, index=False)

print("\nSaved diagnosis table:")
print(diag_path)

print("\nSaved map summary:")
print(map_summary_path)

print("\nSaved scheme summary:")
print(scheme_summary_path)

print("\nCandidate maps generated:")
for rec in scheme_records:
    print(rec["scheme_id"], "->", rec["png_file"])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

analytical_path = process_dir / "df2015_analytical.csv"
country_gini_path = process_dir / "country_gini_2015.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(analytical_path)
country_gini = pd.read_csv(country_gini_path)

print("df shape:", df.shape)
print("country_gini shape:", country_gini.shape)

# =========================
# 2. define variables to check
# =========================
raw_vars = ["SES_raw", "rural", "age", "gender", "education", "Hhsize"]
group_vars = ["ses_group", "rural_group", "age_group", "gender_group", "education_group", "hhsize_group"]

# make sure numeric
for col in raw_vars + group_vars + ["population", "EC2015_pc"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# =========================
# 3. country level completeness and grouping diagnostics
# =========================
country_diag = (
    df.groupby(["analytical_region", "iso_code", "country"])
    .agg(
        n_gdl=("GDLcode", "count"),
        total_population=("population", "sum"),
        mean_ecpc=("EC2015_pc", "mean"),

        ses_raw_nonmissing=("SES_raw", lambda x: x.notna().sum()),
        rural_nonmissing=("rural", lambda x: x.notna().sum()),
        age_nonmissing=("age", lambda x: x.notna().sum()),
        gender_nonmissing=("gender", lambda x: x.notna().sum()),
        education_nonmissing=("education", lambda x: x.notna().sum()),
        hhsize_nonmissing=("Hhsize", lambda x: x.notna().sum()),

        ses_group_n=("ses_group", lambda x: pd.Series(x.dropna()).nunique()),
        rural_group_n=("rural_group", lambda x: pd.Series(x.dropna()).nunique()),
        age_group_n=("age_group", lambda x: pd.Series(x.dropna()).nunique()),
        gender_group_n=("gender_group", lambda x: pd.Series(x.dropna()).nunique()),
        education_group_n=("education_group", lambda x: pd.Series(x.dropna()).nunique()),
        hhsize_group_n=("hhsize_group", lambda x: pd.Series(x.dropna()).nunique()),
    )
    .reset_index()
)

# completeness shares
country_diag["ses_raw_share"] = country_diag["ses_raw_nonmissing"] / country_diag["n_gdl"]
country_diag["rural_share"] = country_diag["rural_nonmissing"] / country_diag["n_gdl"]
country_diag["age_share"] = country_diag["age_nonmissing"] / country_diag["n_gdl"]
country_diag["gender_share"] = country_diag["gender_nonmissing"] / country_diag["n_gdl"]
country_diag["education_share"] = country_diag["education_nonmissing"] / country_diag["n_gdl"]
country_diag["hhsize_share"] = country_diag["hhsize_nonmissing"] / country_diag["n_gdl"]

# whether each attribute has full three group decomposition potential
for col in ["ses_group_n", "rural_group_n", "age_group_n", "gender_group_n", "education_group_n", "hhsize_group_n"]:
    flag_col = col.replace("_n", "_ok")
    country_diag[flag_col] = (country_diag[col] == 3).astype(int)

# total number of attributes with 3 groups
ok_cols = [c for c in country_diag.columns if c.endswith("_ok")]
country_diag["n_attributes_group_ok"] = country_diag[ok_cols].sum(axis=1)

# overall completeness score
share_cols = ["ses_raw_share", "rural_share", "age_share", "gender_share", "education_share", "hhsize_share"]
country_diag["avg_nonmissing_share"] = country_diag[share_cols].mean(axis=1)

# =========================
# 4. merge in country gini
# =========================
country_pool = country_diag.merge(
    country_gini[["iso_code", "country", "analytical_region", "gini"]],
    on=["iso_code", "country", "analytical_region"],
    how="left"
)

# =========================
# 5. initial eligibility rule
#    you can tighten later
# =========================
country_pool["eligible_basic"] = (
    (country_pool["n_gdl"] >= 5) &
    (country_pool["avg_nonmissing_share"] >= 0.95) &
    (country_pool["n_attributes_group_ok"] >= 4)
)

# stricter version
country_pool["eligible_strict"] = (
    (country_pool["n_gdl"] >= 8) &
    (country_pool["avg_nonmissing_share"] >= 0.98) &
    (country_pool["n_attributes_group_ok"] >= 5)
)

# =========================
# 6. rank candidates within each region
#    priority:
#    strict eligibility
#    more attributes usable
#    larger n_gdl
#    higher completeness
# =========================
country_pool = country_pool.sort_values(
    [
        "analytical_region",
        "eligible_strict",
        "eligible_basic",
        "n_attributes_group_ok",
        "n_gdl",
        "avg_nonmissing_share"
    ],
    ascending=[True, False, False, False, False, False]
).reset_index(drop=True)

country_pool["rank_in_region"] = (
    country_pool.groupby("analytical_region")
    .cumcount() + 1
)

# =========================
# 7. create shortlisting tables
# =========================
# top 10 per region for discussion
top10_per_region = (
    country_pool.groupby("analytical_region", group_keys=False)
    .head(10)
    .copy()
)

# one quick automatic pick per region
auto_pick_per_region = (
    country_pool.groupby("analytical_region", group_keys=False)
    .head(1)
    .copy()
)

# =========================
# 8. make a compact discussion table
# =========================
discussion_cols = [
    "analytical_region",
    "rank_in_region",
    "iso_code",
    "country",
    "n_gdl",
    "gini",
    "avg_nonmissing_share",
    "n_attributes_group_ok",
    "eligible_basic",
    "eligible_strict",
    "ses_group_n",
    "rural_group_n",
    "age_group_n",
    "gender_group_n",
    "education_group_n",
    "hhsize_group_n"
]

top10_discussion = top10_per_region[discussion_cols].copy()
auto_pick_discussion = auto_pick_per_region[discussion_cols].copy()

# round for readability
top10_discussion["gini"] = top10_discussion["gini"].round(3)
top10_discussion["avg_nonmissing_share"] = top10_discussion["avg_nonmissing_share"].round(3)

auto_pick_discussion["gini"] = auto_pick_discussion["gini"].round(3)
auto_pick_discussion["avg_nonmissing_share"] = auto_pick_discussion["avg_nonmissing_share"].round(3)

# =========================
# 9. save files
# =========================
country_pool_path = process_dir / "country_candidate_pool.csv"
top10_path = process_dir / "country_top10_per_region_for_discussion.csv"
autopick_path = process_dir / "country_auto_pick_per_region.csv"

country_pool.to_csv(country_pool_path, index=False)
top10_discussion.to_csv(top10_path, index=False)
auto_pick_discussion.to_csv(autopick_path, index=False)

print("\nSaved files:")
print(country_pool_path)
print(top10_path)
print(autopick_path)

# =========================
# 10. print quick outputs
# =========================
print("\nAutomatic one pick per region:")
print(auto_pick_discussion)

print("\nTop 10 candidates per region for discussion:")
print(top10_discussion)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

analytical_path = process_dir / "df2015_analytical.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(analytical_path)

# selected countries
selected_cases = [
    ("Sub-Saharan Africa", "Nigeria"),
    ("Northern Africa and Western Asia", "Egypt"),
    ("South Asia", "India"),
    ("Eastern and South-eastern Asia", "China"),
    ("Europe", "Germany"),
    ("Latin America and the Caribbean", "Brazil"),
    ("Northern America and Oceania", "United States"),
]

# region-consistent colors
case_color_map = {
    "Nigeria": "#b2182b",
    "Egypt": "#ef8a62",
    "India": "#e6ab02",
    "China": "#2166ac",
    "Germany": "#1b7837",
    "Brazil": "#762a83",
    "United States": "#c51b7d",
}

# display labels
case_label_map = {
    "Nigeria": "Nigeria",
    "Egypt": "Egypt",
    "India": "India",
    "China": "China",
    "Germany": "Germany",
    "Brazil": "Brazil",
    "United States": "United States",
}

# make sure numeric
for col in ["population", "EC2015_pc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# valid rows
df_use = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] >= 0)
].copy()

# =========================
# 2. weighted gini
# =========================
def weighted_gini(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan
    if np.all(x == 0):
        return 0.0

    order = np.argsort(x)
    x = x[order]
    w = w[order]

    wx = x * w
    W = w.sum()
    Y = wx.sum()

    if W <= 0 or Y <= 0:
        return np.nan

    p = np.cumsum(w) / W
    L = np.cumsum(wx) / Y

    p_prev = np.concatenate(([0], p[:-1]))
    L_prev = np.concatenate(([0], L[:-1]))

    area = np.sum((L + L_prev) * (p - p_prev) / 2)
    gini = 1 - 2 * area
    return gini

# =========================
# 3. lorenz points
# =========================
def lorenz_points(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return pd.DataFrame(columns=["cum_pop_share", "cum_ec_share"])

    order = np.argsort(x)
    x = x[order]
    w = w[order]

    wx = x * w
    W = w.sum()
    Y = wx.sum()

    if Y <= 0:
        return pd.DataFrame({
            "cum_pop_share": np.concatenate(([0], np.cumsum(w) / W)),
            "cum_ec_share": np.zeros(len(w) + 1)
        })

    return pd.DataFrame({
        "cum_pop_share": np.concatenate(([0], np.cumsum(w) / W)),
        "cum_ec_share": np.concatenate(([0], np.cumsum(wx) / Y))
    })

# =========================
# 4. compute selected-country gini and lorenz
# =========================
country_records = []
lorenz_records = []

for region, country in selected_cases:
    sub = df_use[
        (df_use["analytical_region"] == region) &
        (df_use["country"] == country)
    ].copy()

    if len(sub) == 0:
        print(f"Warning: no data found for {country}")
        continue

    gini_val = weighted_gini(sub["EC2015_pc"], sub["population"])

    country_records.append({
        "analytical_region": region,
        "country": country,
        "n_gdl": len(sub),
        "total_population": sub["population"].sum(),
        "mean_ecpc": np.average(sub["EC2015_pc"], weights=sub["population"]),
        "median_ecpc_unweighted": sub["EC2015_pc"].median(),
        "gini": gini_val
    })

    lor = lorenz_points(sub["EC2015_pc"], sub["population"]).copy()
    lor["country"] = country
    lor["analytical_region"] = region
    lor["point_id"] = np.arange(len(lor))
    lorenz_records.append(lor)

country_gini_selected = pd.DataFrame(country_records)
country_lorenz_selected = pd.concat(lorenz_records, ignore_index=True)

country_gini_selected["gini"] = country_gini_selected["gini"].round(6)

# save intermediate files
country_gini_selected_path = process_dir / "selected_country_gini.csv"
country_lorenz_selected_path = process_dir / "selected_country_lorenz_points.csv"

country_gini_selected.to_csv(country_gini_selected_path, index=False)
country_lorenz_selected.to_csv(country_lorenz_selected_path, index=False)

print("Saved intermediate files:")
print(country_gini_selected_path)
print(country_lorenz_selected_path)

print("\nSelected country Gini summary:")
print(country_gini_selected)

# =========================
# 5. plotting order
#    follow reference style: 4 on top, 3 on bottom
# =========================
plot_order = [
    "Nigeria",
    "Egypt",
    "China",
    "Brazil",
    "India",
    "Germany",
    "United States"
]

# =========================
# 6. plot
# =========================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

fig = plt.figure(figsize=(15.5, 8.4))
gs = fig.add_gridspec(2, 4, hspace=0.30, wspace=0.22)

axes = []
for j in range(4):
    axes.append(fig.add_subplot(gs[0, j]))
for j in range(3):
    axes.append(fig.add_subplot(gs[1, j]))

blank_ax = fig.add_subplot(gs[1, 3])
blank_ax.axis("off")

for i, country in enumerate(plot_order):
    ax = axes[i]

    sub_curve = country_lorenz_selected[country_lorenz_selected["country"] == country].copy()
    sub_info = country_gini_selected[country_gini_selected["country"] == country].copy()

    if sub_curve.empty or sub_info.empty:
        ax.axis("off")
        continue

    color = case_color_map[country]
    gini_val = sub_info["gini"].iloc[0]

    x = sub_curve["cum_pop_share"].values
    y = sub_curve["cum_ec_share"].values

    ax.fill_between(x, y, 0, color=color, alpha=0.18, zorder=1)
    ax.plot(x, y, color=color, linewidth=2.2, solid_capstyle="round", zorder=3)
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.9, color="#9a9a9a", zorder=2)

    ax.set_title(case_label_map[country], pad=8, fontsize=12.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    if i in [0, 4]:
        ax.set_ylabel("Cumulative electricity share", fontsize=10)
    else:
        ax.set_ylabel("")

    if i in [4, 5, 6]:
        ax.set_xlabel("Cumulative population share", fontsize=10)
    else:
        ax.set_xlabel("")

    ax.text(
        0.05, 0.90,
        f"Gini = {gini_val:.3f}",
        transform=ax.transAxes,
        fontsize=10,
        ha="left",
        va="top"
    )

    ax.grid(True, alpha=0.18, linewidth=0.55)

fig.suptitle(
    "Selected country level overall inequality in per capita electricity consumption",
    fontsize=17,
    y=0.98
)

fig.tight_layout(rect=[0, 0, 1, 0.95])

fig_png_path = results_dir / "Selected_country_lorenz_curves.png"
fig_pdf_path = results_dir / "Selected_country_lorenz_curves.pdf"

plt.savefig(fig_png_path, dpi=500, bbox_inches="tight")
plt.savefig(fig_pdf_path, bbox_inches="tight")
plt.show()

print("\nSaved figure files:")
print(fig_png_path)
print(fig_pdf_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

analytical_path = process_dir / "df2015_analytical.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(analytical_path)

# make sure numeric
numeric_cols = [
    "population", "EC2015_pc",
    "ses_group", "rural_group", "age_group",
    "gender_group", "education_group", "hhsize_group"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# valid outcome rows
df = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] > 0) &
    df["analytical_region"].notna()
].copy()

print("Analytical sample shape:", df.shape)

# =========================
# 2. settings
# =========================
region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

attribute_map = {
    "ses_group": "SES",
    "rural_group": "Rural",
    "gender_group": "Gender",
    "age_group": "Age",
    "education_group": "Education",
    "hhsize_group": "Hhsize"
}

# =========================
# 3. weighted Theil T
# =========================
def weighted_theil_t(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (x > 0) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan

    mu = np.sum(w * x) / np.sum(w)
    if mu <= 0:
        return np.nan

    ratio = x / mu
    return np.sum(w * ratio * np.log(ratio)) / np.sum(w)

def theil_decomposition_by_group(data, outcome_col, weight_col, group_col):
    sub = data[[outcome_col, weight_col, group_col]].copy()
    sub = sub.dropna(subset=[outcome_col, weight_col, group_col])
    sub = sub[(sub[outcome_col] > 0) & (sub[weight_col] > 0)].copy()

    if len(sub) == 0:
        return {
            "n_obs": 0,
            "n_groups": 0,
            "Theil_total": np.nan,
            "Theil_between": np.nan,
            "Theil_within": np.nan,
            "between_share": np.nan
        }

    T_total = weighted_theil_t(sub[outcome_col], sub[weight_col])

    W = sub[weight_col].sum()
    mu = np.sum(sub[weight_col] * sub[outcome_col]) / W

    grouped = sub.groupby(group_col)

    between_terms = []
    within_terms = []
    n_groups = 0

    for g, gdf in grouped:
        wg = gdf[weight_col].sum()
        mug = np.sum(gdf[weight_col] * gdf[outcome_col]) / wg

        if wg <= 0 or mug <= 0:
            continue

        n_groups += 1
        sg = wg / W
        rg = mug / mu

        between_terms.append(sg * rg * np.log(rg))

        Tg = weighted_theil_t(gdf[outcome_col], gdf[weight_col])
        within_terms.append(sg * rg * Tg)

    T_between = np.sum(between_terms) if len(between_terms) > 0 else np.nan
    T_within = np.sum(within_terms) if len(within_terms) > 0 else np.nan

    if pd.notna(T_total) and T_total > 0 and pd.notna(T_between):
        between_share = T_between / T_total
    else:
        between_share = np.nan

    return {
        "n_obs": len(sub),
        "n_groups": n_groups,
        "Theil_total": T_total,
        "Theil_between": T_between,
        "Theil_within": T_within,
        "between_share": between_share
    }

# =========================
# 4. run region-level decomposition
# =========================
records = []

for region in region_order:
    dfr = df[df["analytical_region"] == region].copy()

    if len(dfr) == 0:
        continue

    for group_col, attr_label in attribute_map.items():
        res = theil_decomposition_by_group(
            data=dfr,
            outcome_col="EC2015_pc",
            weight_col="population",
            group_col=group_col
        )

        records.append({
            "analytical_region": region,
            "attribute": attr_label,
            "group_col": group_col,
            **res
        })

region_theil = pd.DataFrame(records)

# round for display
for col in ["Theil_total", "Theil_between", "Theil_within", "between_share"]:
    region_theil[col] = region_theil[col].round(6)

# =========================
# 5. panel A data: grouped bar of between_share
# =========================
panelA = region_theil.pivot(
    index="analytical_region",
    columns="attribute",
    values="between_share"
).reset_index()

panelA["analytical_region"] = pd.Categorical(
    panelA["analytical_region"],
    categories=region_order,
    ordered=True
)
panelA = panelA.sort_values("analytical_region").reset_index(drop=True)

# =========================
# 6. panel B data: between vs within stacked bars
#    long format and wide format
# =========================
panelB_long = region_theil[[
    "analytical_region", "attribute", "Theil_between", "Theil_within"
]].copy()

panelB_between = panelB_long.pivot(
    index="analytical_region",
    columns="attribute",
    values="Theil_between"
).reset_index()

panelB_within = panelB_long.pivot(
    index="analytical_region",
    columns="attribute",
    values="Theil_within"
).reset_index()

for d in [panelB_between, panelB_within]:
    d["analytical_region"] = pd.Categorical(
        d["analytical_region"],
        categories=region_order,
        ordered=True
    )
    d.sort_values("analytical_region", inplace=True)
    d.reset_index(drop=True, inplace=True)

# =========================
# 7. save
# =========================
region_theil_path = process_dir / "region_theil_decomposition.csv"
panelA_path = process_dir / "Figure2_panelA_between_share.csv"
panelB_between_path = process_dir / "Figure2_panelB_between.csv"
panelB_within_path = process_dir / "Figure2_panelB_within.csv"
panelB_long_path = process_dir / "Figure2_panelB_long.csv"

region_theil.to_csv(region_theil_path, index=False)
panelA.to_csv(panelA_path, index=False)
panelB_between.to_csv(panelB_between_path, index=False)
panelB_within.to_csv(panelB_within_path, index=False)
panelB_long.to_csv(panelB_long_path, index=False)

print("Saved files:")
print(region_theil_path)
print(panelA_path)
print(panelB_between_path)
print(panelB_within_path)
print(panelB_long_path)

print("\nRegion-level Theil decomposition:")
print(region_theil)

print("\nPanel A preview:")
print(panelA)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

analytical_path = process_dir / "df2015_analytical.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(analytical_path)

# selected countries
selected_cases = [
    ("Sub-Saharan Africa", "Nigeria"),
    ("Northern Africa and Western Asia", "Egypt"),
    ("South Asia", "India"),
    ("Eastern and South-eastern Asia", "China"),
    ("Europe", "Germany"),
    ("Latin America and the Caribbean", "Brazil"),
    ("Northern America and Oceania", "United States"),
]

# numeric
numeric_cols = [
    "population", "EC2015_pc",
    "ses_group", "rural_group", "age_group",
    "gender_group", "education_group", "hhsize_group"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] > 0)
].copy()

# =========================
# 2. weighted Theil T
# =========================
def weighted_theil_t(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (x > 0) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan

    mu = np.sum(w * x) / np.sum(w)
    if mu <= 0:
        return np.nan

    ratio = x / mu
    return np.sum(w * ratio * np.log(ratio)) / np.sum(w)

def theil_decomposition_by_group(data, outcome_col, weight_col, group_col):
    sub = data[[outcome_col, weight_col, group_col]].copy()
    sub = sub.dropna(subset=[outcome_col, weight_col, group_col])
    sub = sub[(sub[outcome_col] > 0) & (sub[weight_col] > 0)].copy()

    if len(sub) == 0:
        return {
            "n_obs": 0,
            "n_groups": 0,
            "Theil_total": np.nan,
            "Theil_between": np.nan,
            "Theil_within": np.nan,
            "between_share": np.nan
        }

    T_total = weighted_theil_t(sub[outcome_col], sub[weight_col])

    W = sub[weight_col].sum()
    mu = np.sum(sub[weight_col] * sub[outcome_col]) / W

    grouped = sub.groupby(group_col)

    between_terms = []
    within_terms = []
    n_groups = 0

    for g, gdf in grouped:
        wg = gdf[weight_col].sum()
        mug = np.sum(gdf[weight_col] * gdf[outcome_col]) / wg

        if wg <= 0 or mug <= 0:
            continue

        n_groups += 1
        sg = wg / W
        rg = mug / mu

        between_terms.append(sg * rg * np.log(rg))

        Tg = weighted_theil_t(gdf[outcome_col], gdf[weight_col])
        within_terms.append(sg * rg * Tg)

    T_between = np.sum(between_terms) if len(between_terms) > 0 else np.nan
    T_within = np.sum(within_terms) if len(within_terms) > 0 else np.nan
    between_share = T_between / T_total if pd.notna(T_total) and T_total > 0 and pd.notna(T_between) else np.nan

    return {
        "n_obs": len(sub),
        "n_groups": n_groups,
        "Theil_total": T_total,
        "Theil_between": T_between,
        "Theil_within": T_within,
        "between_share": between_share
    }

# =========================
# 3. run selected-country decomposition
# =========================
attribute_map = {
    "ses_group": "SES",
    "rural_group": "Rural",
    "gender_group": "Gender",
    "age_group": "Age",
    "education_group": "Education",
    "hhsize_group": "Hhsize"
}

records = []

for region, country in selected_cases:
    dfi = df[(df["analytical_region"] == region) & (df["country"] == country)].copy()

    for group_col, attr_label in attribute_map.items():
        res = theil_decomposition_by_group(
            data=dfi,
            outcome_col="EC2015_pc",
            weight_col="population",
            group_col=group_col
        )

        records.append({
            "analytical_region": region,
            "country": country,
            "attribute": attr_label,
            "group_col": group_col,
            **res
        })

country_theil = pd.DataFrame(records)

for col in ["Theil_total", "Theil_between", "Theil_within", "between_share"]:
    country_theil[col] = country_theil[col].round(6)

country_theil_path = process_dir / "selected_country_theil_decomposition.csv"
country_theil.to_csv(country_theil_path, index=False)

print("Saved:")
print(country_theil_path)

print("\nSelected-country decomposition:")
print(country_theil)

# =========================
# 4. Panel A data
# =========================
country_order = ["Nigeria", "Egypt", "India", "China", "Germany", "Brazil", "United States"]
attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize"]

panelA = country_theil.pivot(index="country", columns="attribute", values="between_share").reindex(country_order)

panelA_path = process_dir / "Figure3_panelA_selected_country_between_share.csv"
panelA.to_csv(panelA_path)

print("\nSaved:")
print(panelA_path)

# =========================
# 5. plot Panel A only
# =========================
attr_colors = {
    "SES": "#b2182b",
    "Rural": "#ef8a62",
    "Gender": "#fddbc7",
    "Age": "#67a9cf",
    "Education": "#2166ac",
    "Hhsize": "#762a83"
}

country_labels = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United States"
]

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10

fig, ax = plt.subplots(figsize=(15, 6.5))

x = np.arange(len(country_order))
n_attr = len(attr_order)
bar_width = 0.12
offsets = (np.arange(n_attr) - (n_attr - 1) / 2) * bar_width

for i, attr in enumerate(attr_order):
    vals = panelA[attr].values
    ax.bar(
        x + offsets[i],
        vals,
        width=bar_width,
        label=attr,
        color=attr_colors[attr],
        edgecolor="white",
        linewidth=0.6
    )

# label row max only
for r_idx, country in enumerate(country_order):
    row = panelA.loc[country]
    max_attr = row.idxmax()
    max_val = row.max()
    if pd.notna(max_val):
        i = attr_order.index(max_attr)
        xpos = x[r_idx] + offsets[i]
        ax.text(
            xpos, max_val + 0.006,
            f"{max_val:.2f}",
            ha="center", va="bottom", fontsize=9
        )

ax.set_title("Selected country level between group share of total inequality", pad=10)
ax.set_ylabel("Between share")
ax.set_xticks(x)
ax.set_xticklabels(country_labels)
ax.set_ylim(0, max(0.05, np.nanmax(panelA.values) * 1.25))
ax.grid(axis="y", alpha=0.2, linewidth=0.6)
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.18), frameon=False)

fig.tight_layout(rect=[0, 0, 1, 0.94])

fig_png_path = results_dir / "Figure3_selected_country_between_share.png"
fig_pdf_path = results_dir / "Figure3_selected_country_between_share.pdf"

plt.savefig(fig_png_path, dpi=500, bbox_inches="tight")
plt.savefig(fig_pdf_path, bbox_inches="tight")
plt.show()

print(fig_png_path)
print(fig_pdf_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

region_theil_path = process_dir / "region_theil_decomposition.csv"

df = pd.read_csv(region_theil_path)

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]
attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize"]

df["analytical_region"] = pd.Categorical(df["analytical_region"], categories=region_order, ordered=True)
df["attribute"] = pd.Categorical(df["attribute"], categories=attr_order, ordered=True)
df = df.sort_values(["analytical_region", "attribute"]).reset_index(drop=True)

panelA = df.pivot(index="analytical_region", columns="attribute", values="between_share").reindex(region_order)

region_short_labels = [
    "Sub-Saharan\nAfrica",
    "Northern Africa\nand Western Asia",
    "South Asia",
    "Eastern and\nSouth-eastern Asia",
    "Europe",
    "Latin America\nand the Caribbean",
    "Northern America\nand Oceania"
]

attr_colors = {
    "SES": "#b2182b",
    "Rural": "#ef8a62",
    "Gender": "#fddbc7",
    "Age": "#67a9cf",
    "Education": "#2166ac",
    "Hhsize": "#762a83"
}

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10

fig, ax = plt.subplots(figsize=(15, 6.8))

x = np.arange(len(region_order))
n_attr = len(attr_order)
bar_width = 0.12
offsets = (np.arange(n_attr) - (n_attr - 1) / 2) * bar_width

for i, attr in enumerate(attr_order):
    vals = panelA[attr].values
    ax.bar(
        x + offsets[i],
        vals,
        width=bar_width,
        label=attr,
        color=attr_colors[attr],
        edgecolor="white",
        linewidth=0.6
    )

# 每个 region 标注最大值
for r_idx, region in enumerate(region_order):
    row = panelA.loc[region]
    max_attr = row.idxmax()
    max_val = row.max()
    if pd.notna(max_val):
        i = attr_order.index(max_attr)
        xpos = x[r_idx] + offsets[i]
        ax.text(
            xpos,
            max_val + 0.006,
            f"{max_val:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax.set_title("Region level between group share of total inequality", pad=10)
ax.set_ylabel("Between share")
ax.set_xticks(x)
ax.set_xticklabels(region_short_labels)
ax.set_ylim(0, max(0.05, np.nanmax(panelA.values) * 1.25))
ax.grid(axis="y", alpha=0.2, linewidth=0.6)
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.16), frameon=False)

fig.tight_layout(rect=[0, 0, 1, 0.94])

fig_png_path = results_dir / "Figure2_region_between_share_only.png"
fig_pdf_path = results_dir / "Figure2_region_between_share_only.pdf"

plt.savefig(fig_png_path, dpi=500, bbox_inches="tight")
plt.savefig(fig_pdf_path, bbox_inches="tight")
plt.show()

print(fig_png_path)
print(fig_pdf_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

region_theil_path = process_dir / "region_theil_decomposition.csv"
country_theil_path = process_dir / "selected_country_theil_decomposition.csv"

# =========================
# 1. read data
# =========================
region_df = pd.read_csv(region_theil_path)
country_df = pd.read_csv(country_theil_path)

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

country_order = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United States"
]

attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize"]

region_df["analytical_region"] = pd.Categorical(region_df["analytical_region"], categories=region_order, ordered=True)
region_df["attribute"] = pd.Categorical(region_df["attribute"], categories=attr_order, ordered=True)
region_df = region_df.sort_values(["analytical_region", "attribute"]).reset_index(drop=True)

country_df["country"] = pd.Categorical(country_df["country"], categories=country_order, ordered=True)
country_df["attribute"] = pd.Categorical(country_df["attribute"], categories=attr_order, ordered=True)
country_df = country_df.sort_values(["country", "attribute"]).reset_index(drop=True)

region_panel = region_df.pivot(index="analytical_region", columns="attribute", values="between_share").reindex(region_order)
country_panel = country_df.pivot(index="country", columns="attribute", values="between_share").reindex(country_order)

# =========================
# 2. unified style
# =========================
attr_colors = {
    "SES": "#b2182b",
    "Rural": "#ef8a62",
    "Gender": "#fddbc7",
    "Age": "#67a9cf",
    "Education": "#2166ac",
    "Hhsize": "#762a83"
}

region_labels = [
    "Sub-Saharan\nAfrica",
    "Northern Africa\nand Western Asia",
    "South Asia",
    "Eastern and\nSouth-eastern Asia",
    "Europe",
    "Latin America\nand the Caribbean",
    "Northern America\nand Oceania"
]

country_labels = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United\nStates"
]

# unified y-axis upper bound
global_max = np.nanmax([
    np.nanmax(region_panel.values),
    np.nanmax(country_panel.values)
])
ymax = max(0.05, global_max * 1.28)

# unified plotting params
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.9
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["legend.fontsize"] = 15

# =========================
# 3. helper function
# =========================
def plot_between_share(panel_df, xlabels, title, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(16, 7.8))

    x = np.arange(len(panel_df.index))
    n_attr = len(attr_order)
    bar_width = 0.12
    offsets = (np.arange(n_attr) - (n_attr - 1) / 2) * bar_width

    for i, attr in enumerate(attr_order):
        vals = panel_df[attr].values
        ax.bar(
            x + offsets[i],
            vals,
            width=bar_width,
            label=attr,
            color=attr_colors[attr],
            edgecolor="white",
            linewidth=0.7
        )

    # annotate max within each row
    for r_idx, idx in enumerate(panel_df.index):
        row = panel_df.loc[idx]
        max_attr = row.idxmax()
        max_val = row.max()
        if pd.notna(max_val):
            i = attr_order.index(max_attr)
            xpos = x[r_idx] + offsets[i]
            ax.text(
                xpos,
                max_val + ymax * 0.018,
                f"{max_val:.2f}",
                ha="center",
                va="bottom",
                fontsize=18
            )

    ax.set_title(title, pad=14)
    ax.set_ylabel("Between share")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7)
    ax.set_axisbelow(True)

    ax.legend(
        ncol=6,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        frameon=False
    )

    fig.tight_layout(rect=[0, 0, 1, 0.93])

    plt.savefig(out_png, dpi=500, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

# =========================
# 4. plot region
# =========================
region_png = results_dir / "Figure2_region_between_share_only_matched_large_labels.png"
region_pdf = results_dir / "Figure2_region_between_share_only_matched_large_labels.pdf"

plot_between_share(
    panel_df=region_panel,
    xlabels=region_labels,
    title="Region level between group share of total inequality",
    out_png=region_png,
    out_pdf=region_pdf
)

# =========================
# 5. plot country
# =========================
country_png = results_dir / "Figure3_selected_country_between_share_matched_large_labels.png"
country_pdf = results_dir / "Figure3_selected_country_between_share_matched_large_labels.pdf"

plot_between_share(
    panel_df=country_panel,
    xlabels=country_labels,
    title="Selected country level between group share of total inequality",
    out_png=country_png,
    out_pdf=country_pdf
)

print(region_png)
print(region_pdf)
print(country_png)
print(country_pdf)
print(f"Unified y-axis max = {ymax:.3f}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(input_path)

selected_cases = [
    ("Sub-Saharan Africa", "Nigeria"),
    ("Northern Africa and Western Asia", "Egypt"),
    ("South Asia", "India"),
    ("Eastern and South-eastern Asia", "China"),
    ("Europe", "Germany"),
    ("Latin America and the Caribbean", "Brazil"),
    ("Northern America and Oceania", "United States"),
]

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

country_order = ["Nigeria", "Egypt", "India", "China", "Germany", "Brazil", "United States"]

# numeric
for col in ["population", "EC2015_pc", "ses_group", "rural_group"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep valid rows
df = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    df["ses_group"].notna() &
    df["rural_group"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] > 0)
].copy()

print("Analytical sample shape:", df.shape)

# =========================
# 2. build cross group
# =========================
# 3 x 3 -> up to 9 groups
df["ses_rural_group"] = (
    df["ses_group"].astype(int).astype(str) + "_" +
    df["rural_group"].astype(int).astype(str)
)

# =========================
# 3. weighted Theil T
# =========================
def weighted_theil_t(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (x > 0) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan

    mu = np.sum(w * x) / np.sum(w)
    if mu <= 0:
        return np.nan

    ratio = x / mu
    return np.sum(w * ratio * np.log(ratio)) / np.sum(w)

def theil_decomposition_by_group(data, outcome_col, weight_col, group_col):
    sub = data[[outcome_col, weight_col, group_col]].copy()
    sub = sub.dropna(subset=[outcome_col, weight_col, group_col])
    sub = sub[(sub[outcome_col] > 0) & (sub[weight_col] > 0)].copy()

    if len(sub) == 0:
        return {
            "n_obs": 0,
            "n_groups": 0,
            "Theil_total": np.nan,
            "Theil_between": np.nan,
            "Theil_within": np.nan,
            "between_share": np.nan
        }

    T_total = weighted_theil_t(sub[outcome_col], sub[weight_col])

    W = sub[weight_col].sum()
    mu = np.sum(sub[weight_col] * sub[outcome_col]) / W

    between_terms = []
    within_terms = []
    n_groups = 0

    for g, gdf in sub.groupby(group_col):
        wg = gdf[weight_col].sum()
        mug = np.sum(gdf[weight_col] * gdf[outcome_col]) / wg

        if wg <= 0 or mug <= 0:
            continue

        n_groups += 1
        sg = wg / W
        rg = mug / mu

        between_terms.append(sg * rg * np.log(rg))
        Tg = weighted_theil_t(gdf[outcome_col], gdf[weight_col])
        within_terms.append(sg * rg * Tg)

    T_between = np.sum(between_terms) if len(between_terms) > 0 else np.nan
    T_within = np.sum(within_terms) if len(within_terms) > 0 else np.nan
    between_share = T_between / T_total if pd.notna(T_total) and T_total > 0 and pd.notna(T_between) else np.nan

    return {
        "n_obs": len(sub),
        "n_groups": n_groups,
        "Theil_total": T_total,
        "Theil_between": T_between,
        "Theil_within": T_within,
        "between_share": between_share
    }

# =========================
# 4. helper for comparison
# =========================
def compare_single_and_cross(data, level_name, level_value, level_type):
    ses_res = theil_decomposition_by_group(data, "EC2015_pc", "population", "ses_group")
    rural_res = theil_decomposition_by_group(data, "EC2015_pc", "population", "rural_group")
    cross_res = theil_decomposition_by_group(data, "EC2015_pc", "population", "ses_rural_group")

    best_single = np.nanmax([ses_res["between_share"], rural_res["between_share"]])
    delta_vs_best_single = cross_res["between_share"] - best_single if pd.notna(best_single) and pd.notna(cross_res["between_share"]) else np.nan
    delta_vs_ses = cross_res["between_share"] - ses_res["between_share"] if pd.notna(ses_res["between_share"]) and pd.notna(cross_res["between_share"]) else np.nan
    delta_vs_rural = cross_res["between_share"] - rural_res["between_share"] if pd.notna(rural_res["between_share"]) and pd.notna(cross_res["between_share"]) else np.nan

    return {
        "level_type": level_type,
        "level_name": level_name,
        "level_value": level_value,

        "ses_n_groups": ses_res["n_groups"],
        "rural_n_groups": rural_res["n_groups"],
        "cross_n_groups": cross_res["n_groups"],

        "ses_between_share": ses_res["between_share"],
        "rural_between_share": rural_res["between_share"],
        "cross_between_share": cross_res["between_share"],

        "ses_Theil_between": ses_res["Theil_between"],
        "rural_Theil_between": rural_res["Theil_between"],
        "cross_Theil_between": cross_res["Theil_between"],

        "best_single_between_share": best_single,
        "delta_cross_vs_best_single": delta_vs_best_single,
        "delta_cross_vs_ses": delta_vs_ses,
        "delta_cross_vs_rural": delta_vs_rural,
    }

# =========================
# 5. region level
# =========================
region_records = []

for region in region_order:
    sub = df[df["analytical_region"] == region].copy()
    if len(sub) == 0:
        continue
    region_records.append(
        compare_single_and_cross(
            data=sub,
            level_name="analytical_region",
            level_value=region,
            level_type="region"
        )
    )

region_cross = pd.DataFrame(region_records)

# =========================
# 6. selected country level
# =========================
country_records = []

for region, country in selected_cases:
    sub = df[(df["analytical_region"] == region) & (df["country"] == country)].copy()
    if len(sub) == 0:
        continue
    country_records.append(
        compare_single_and_cross(
            data=sub,
            level_name="country",
            level_value=country,
            level_type="country"
        )
    )

country_cross = pd.DataFrame(country_records)

# =========================
# 7. round and save
# =========================
round_cols = [
    "ses_between_share", "rural_between_share", "cross_between_share",
    "ses_Theil_between", "rural_Theil_between", "cross_Theil_between",
    "best_single_between_share", "delta_cross_vs_best_single",
    "delta_cross_vs_ses", "delta_cross_vs_rural"
]

for d in [region_cross, country_cross]:
    for col in round_cols:
        d[col] = d[col].round(6)

region_out = process_dir / "region_cross_ses_rural_comparison.csv"
country_out = process_dir / "selected_country_cross_ses_rural_comparison.csv"

region_cross.to_csv(region_out, index=False)
country_cross.to_csv(country_out, index=False)

print("Saved files:")
print(region_out)
print(country_out)

print("\nRegion level comparison:")
print(region_cross[[
    "level_value",
    "ses_between_share",
    "rural_between_share",
    "cross_between_share",
    "best_single_between_share",
    "delta_cross_vs_best_single",
    "cross_n_groups"
]])

print("\nSelected country level comparison:")
print(country_cross[[
    "level_value",
    "ses_between_share",
    "rural_between_share",
    "cross_between_share",
    "best_single_between_share",
    "delta_cross_vs_best_single",
    "cross_n_groups"
]])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import itertools
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"
region_theil_path = process_dir / "region_theil_decomposition.csv"
country_theil_path = process_dir / "selected_country_theil_decomposition.csv"

# =========================
# 1. read data
# =========================
df = pd.read_csv(input_path)
region_df = pd.read_csv(region_theil_path)
country_df = pd.read_csv(country_theil_path)

selected_cases = [
    ("Sub-Saharan Africa", "Nigeria"),
    ("Northern Africa and Western Asia", "Egypt"),
    ("South Asia", "India"),
    ("Eastern and South-eastern Asia", "China"),
    ("Europe", "Germany"),
    ("Latin America and the Caribbean", "Brazil"),
    ("Northern America and Oceania", "United States"),
]

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

country_order = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United States"
]

attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize"]
group_map = {
    "SES": "ses_group",
    "Rural": "rural_group",
    "Gender": "gender_group",
    "Age": "age_group",
    "Education": "education_group",
    "Hhsize": "hhsize_group"
}

# numeric
for col in ["population", "EC2015_pc"] + list(group_map.values()):
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["population"].notna() &
    df["EC2015_pc"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] > 0)
].copy()

# =========================
# 2. Theil functions
# =========================
def weighted_theil_t(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (x > 0) & (w > 0)
    x = x[mask]
    w = w[mask]
    if len(x) == 0:
        return np.nan
    mu = np.sum(w * x) / np.sum(w)
    if mu <= 0:
        return np.nan
    ratio = x / mu
    return np.sum(w * ratio * np.log(ratio)) / np.sum(w)

def theil_decomposition_by_group(data, outcome_col, weight_col, group_col):
    sub = data[[outcome_col, weight_col, group_col]].copy()
    sub = sub.dropna(subset=[outcome_col, weight_col, group_col])
    sub = sub[(sub[outcome_col] > 0) & (sub[weight_col] > 0)].copy()

    if len(sub) == 0:
        return {
            "n_obs": 0,
            "n_groups": 0,
            "Theil_total": np.nan,
            "Theil_between": np.nan,
            "Theil_within": np.nan,
            "between_share": np.nan
        }

    T_total = weighted_theil_t(sub[outcome_col], sub[weight_col])

    W = sub[weight_col].sum()
    mu = np.sum(sub[weight_col] * sub[outcome_col]) / W

    between_terms = []
    within_terms = []
    n_groups = 0

    for _, gdf in sub.groupby(group_col):
        wg = gdf[weight_col].sum()
        mug = np.sum(gdf[weight_col] * gdf[outcome_col]) / wg
        if wg <= 0 or mug <= 0:
            continue
        n_groups += 1
        sg = wg / W
        rg = mug / mu
        between_terms.append(sg * rg * np.log(rg))
        Tg = weighted_theil_t(gdf[outcome_col], gdf[weight_col])
        within_terms.append(sg * rg * Tg)

    T_between = np.sum(between_terms) if len(between_terms) > 0 else np.nan
    T_within = np.sum(within_terms) if len(within_terms) > 0 else np.nan
    between_share = T_between / T_total if pd.notna(T_total) and T_total > 0 and pd.notna(T_between) else np.nan

    return {
        "n_obs": len(sub),
        "n_groups": n_groups,
        "Theil_total": T_total,
        "Theil_between": T_between,
        "Theil_within": T_within,
        "between_share": between_share
    }

# =========================
# 3. build all pairwise cross groups
# =========================
pair_defs = list(itertools.combinations(attr_order, 2))

for a1, a2 in pair_defs:
    c1 = group_map[a1]
    c2 = group_map[a2]
    new_col = f"cross__{a1}__{a2}"
    df[new_col] = (
        df[c1].astype("Int64").astype(str) + "_" +
        df[c2].astype("Int64").astype(str)
    )

# =========================
# 4. evaluate all pairwise crosses
# =========================
def evaluate_best_cross(data, level_type, level_value):
    rows = []
    for a1, a2 in pair_defs:
        cross_col = f"cross__{a1}__{a2}"
        res = theil_decomposition_by_group(data, "EC2015_pc", "population", cross_col)
        rows.append({
            "level_type": level_type,
            "level_value": level_value,
            "pair_name": f"{a1} × {a2}",
            "pair_code": f"{a1}__{a2}",
            "cross_group_col": cross_col,
            "n_groups": res["n_groups"],
            "between_share": res["between_share"],
            "Theil_between": res["Theil_between"],
            "Theil_total": res["Theil_total"]
        })
    out = pd.DataFrame(rows).sort_values("between_share", ascending=False).reset_index(drop=True)
    return out

region_cross_all = []
for region in region_order:
    sub = df[df["analytical_region"] == region].copy()
    if len(sub) == 0:
        continue
    region_cross_all.append(evaluate_best_cross(sub, "region", region))
region_cross_all = pd.concat(region_cross_all, ignore_index=True)

country_cross_all = []
for region, country in selected_cases:
    sub = df[(df["analytical_region"] == region) & (df["country"] == country)].copy()
    if len(sub) == 0:
        continue
    country_cross_all.append(evaluate_best_cross(sub, "country", country))
country_cross_all = pd.concat(country_cross_all, ignore_index=True)

for d in [region_cross_all, country_cross_all]:
    for col in ["between_share", "Theil_between", "Theil_total"]:
        d[col] = d[col].round(6)

region_cross_all_path = process_dir / "region_all_pairwise_cross_results.csv"
country_cross_all_path = process_dir / "selected_country_all_pairwise_cross_results.csv"
region_cross_all.to_csv(region_cross_all_path, index=False)
country_cross_all.to_csv(country_cross_all_path, index=False)

# best only
region_best_cross = region_cross_all.groupby("level_value", as_index=False).first()
country_best_cross = country_cross_all.groupby("level_value", as_index=False).first()

region_best_cross_path = process_dir / "region_best_pairwise_cross.csv"
country_best_cross_path = process_dir / "selected_country_best_pairwise_cross.csv"
region_best_cross.to_csv(region_best_cross_path, index=False)
country_best_cross.to_csv(country_best_cross_path, index=False)

print("Saved:")
print(region_cross_all_path)
print(country_cross_all_path)
print(region_best_cross_path)
print(country_best_cross_path)

print("\nBest cross by region:")
print(region_best_cross[["level_value", "pair_name", "between_share", "n_groups"]])

print("\nBest cross by selected country:")
print(country_best_cross[["level_value", "pair_name", "between_share", "n_groups"]])

# =========================
# 5. build panels for plotting
# =========================
region_df["analytical_region"] = pd.Categorical(region_df["analytical_region"], categories=region_order, ordered=True)
region_df["attribute"] = pd.Categorical(region_df["attribute"], categories=attr_order, ordered=True)
region_df = region_df.sort_values(["analytical_region", "attribute"]).reset_index(drop=True)

region_panel = region_df.pivot(index="analytical_region", columns="attribute", values="between_share").reindex(region_order)
region_panel["Best cross"] = region_best_cross.set_index("level_value").reindex(region_order)["between_share"].values

country_df["country"] = pd.Categorical(country_df["country"], categories=country_order, ordered=True)
country_df["attribute"] = pd.Categorical(country_df["attribute"], categories=attr_order, ordered=True)
country_df = country_df.sort_values(["country", "attribute"]).reset_index(drop=True)

country_panel = country_df.pivot(index="country", columns="attribute", values="between_share").reindex(country_order)
country_panel["Best cross"] = country_best_cross.set_index("level_value").reindex(country_order)["between_share"].values

plot_attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize", "Best cross"]

# =========================
# 6. style
# =========================
attr_colors = {
    "SES": "#b2182b",
    "Rural": "#ef8a62",
    "Gender": "#fddbc7",
    "Age": "#67a9cf",
    "Education": "#2166ac",
    "Hhsize": "#762a83",
    "Best cross": "#7f7f7f"
}

region_labels = [
    "Sub-Saharan\nAfrica",
    "Northern Africa\nand Western Asia",
    "South Asia",
    "Eastern and\nSouth-eastern Asia",
    "Europe",
    "Latin America\nand the Caribbean",
    "Northern America\nand Oceania"
]

country_labels = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United\nStates"
]

global_max = np.nanmax([
    np.nanmax(region_panel.values),
    np.nanmax(country_panel.values)
])
ymax = max(0.05, global_max * 1.22)

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.9
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["legend.fontsize"] = 14

# =========================
# 7. plot helper
# =========================
def plot_panel(panel_df, best_cross_df, xlabels, title, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(16, 7.8))

    x = np.arange(len(panel_df.index))
    n_attr = len(plot_attr_order)
    bar_width = 0.105
    offsets = (np.arange(n_attr) - (n_attr - 1) / 2) * bar_width

    for i, attr in enumerate(plot_attr_order):
        vals = panel_df[attr].values
        if attr == "Best cross":
            ax.bar(
                x + offsets[i],
                vals,
                width=bar_width,
                label=attr,
                color=attr_colors[attr],
                edgecolor="#4d4d4d",
                linewidth=0.9,
                hatch="///"
            )
        else:
            ax.bar(
                x + offsets[i],
                vals,
                width=bar_width,
                label=attr,
                color=attr_colors[attr],
                edgecolor="white",
                linewidth=0.7
            )

    # top 2 labels
    for r_idx, idx in enumerate(panel_df.index):
        row = panel_df.loc[idx].dropna().sort_values(ascending=False)
        top_vals = row.iloc[:2]
        for rank, (attr_name, val) in enumerate(top_vals.items()):
            i = plot_attr_order.index(attr_name)
            xpos = x[r_idx] + offsets[i]
            extra = ymax * (0.015 + rank * 0.03)

            if attr_name == "Best cross":
                pair_name = best_cross_df.set_index("level_value").loc[idx, "pair_name"]
                label_text = f"{pair_name}\n{val:.2f}"
            else:
                label_text = f"{val:.2f}"

            ax.text(
                xpos,
                val + extra,
                label_text,
                ha="center",
                va="bottom",
                fontsize=16
            )

    ax.set_title(title, pad=14)
    ax.set_ylabel("Between share")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7)
    ax.set_axisbelow(True)

    ax.legend(
        ncol=7,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.20),
        frameon=False,
        columnspacing=1.2,
        handlelength=1.8
    )

    fig.tight_layout(rect=[0, 0, 1, 0.90])

    plt.savefig(out_png, dpi=500, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

# =========================
# 8. region figure
# =========================
region_png = results_dir / "Figure2_region_between_share_with_best_cross.png"
region_pdf = results_dir / "Figure2_region_between_share_with_best_cross.pdf"

plot_panel(
    panel_df=region_panel[plot_attr_order],
    best_cross_df=region_best_cross,
    xlabels=region_labels,
    title="Region level between group share of total inequality",
    out_png=region_png,
    out_pdf=region_pdf
)

# =========================
# 9. country figure
# =========================
country_png = results_dir / "Figure3_selected_country_between_share_with_best_cross.png"
country_pdf = results_dir / "Figure3_selected_country_between_share_with_best_cross.pdf"

plot_panel(
    panel_df=country_panel[plot_attr_order],
    best_cross_df=country_best_cross,
    xlabels=country_labels,
    title="Selected country level between group share of total inequality",
    out_png=country_png,
    out_pdf=country_pdf
)

print(region_png)
print(region_pdf)
print(country_png)
print(country_pdf)
print(f"Unified y-axis max = {ymax:.3f}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

region_theil_path = process_dir / "region_theil_decomposition.csv"
country_theil_path = process_dir / "selected_country_theil_decomposition.csv"
region_best_cross_path = process_dir / "region_best_pairwise_cross.csv"
country_best_cross_path = process_dir / "selected_country_best_pairwise_cross.csv"

# =========================
# 1. read data
# =========================
region_df = pd.read_csv(region_theil_path)
country_df = pd.read_csv(country_theil_path)
region_best_cross = pd.read_csv(region_best_cross_path)
country_best_cross = pd.read_csv(country_best_cross_path)

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

country_order = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United States"
]

attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize"]
plot_attr_order = ["SES", "Rural", "Gender", "Age", "Education", "Hhsize", "Best cross"]

# =========================
# 2. build panels
# =========================
region_df["analytical_region"] = pd.Categorical(region_df["analytical_region"], categories=region_order, ordered=True)
region_df["attribute"] = pd.Categorical(region_df["attribute"], categories=attr_order, ordered=True)
region_df = region_df.sort_values(["analytical_region", "attribute"]).reset_index(drop=True)

region_panel = region_df.pivot(index="analytical_region", columns="attribute", values="between_share").reindex(region_order)
region_panel["Best cross"] = region_best_cross.set_index("level_value").reindex(region_order)["between_share"].values
region_panel = region_panel[plot_attr_order]

country_df["country"] = pd.Categorical(country_df["country"], categories=country_order, ordered=True)
country_df["attribute"] = pd.Categorical(country_df["attribute"], categories=attr_order, ordered=True)
country_df = country_df.sort_values(["country", "attribute"]).reset_index(drop=True)

country_panel = country_df.pivot(index="country", columns="attribute", values="between_share").reindex(country_order)
country_panel["Best cross"] = country_best_cross.set_index("level_value").reindex(country_order)["between_share"].values
country_panel = country_panel[plot_attr_order]

# =========================
# 3. style
# =========================
attr_colors = {
    "SES": "#b2182b",
    "Rural": "#ef8a62",
    "Gender": "#fddbc7",
    "Age": "#67a9cf",
    "Education": "#2166ac",
    "Hhsize": "#762a83",
    "Best cross": "#cfcfcf"
}

region_labels = [
    "Sub-Saharan\nAfrica",
    "Northern Africa\nand Western Asia",
    "South Asia",
    "Eastern and\nSouth-eastern Asia",
    "Europe",
    "Latin America\nand the Caribbean",
    "Northern America\nand Oceania"
]

country_labels = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Germany",
    "Brazil",
    "United\nStates"
]

global_max = np.nanmax([
    np.nanmax(region_panel.values),
    np.nanmax(country_panel.values)
])
ymax = max(0.05, global_max * 1.22)

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.9
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["legend.fontsize"] = 14

# =========================
# 4. helper
# =========================
def split_pair_name(pair_name: str) -> str:
    if " × " in pair_name:
        a, b = pair_name.split(" × ")
        return f"{a}\n× {b}"
    return pair_name

def plot_panel(panel_df, best_cross_df, xlabels, title, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(16, 7.8))

    x = np.arange(len(panel_df.index))
    regular_width = 0.11
    cross_width = 0.075
    offsets = (np.arange(len(plot_attr_order)) - (len(plot_attr_order) - 1) / 2) * regular_width

    for i, attr in enumerate(plot_attr_order):
        vals = panel_df[attr].values

        if attr == "Best cross":
            ax.bar(
                x + offsets[i],
                vals,
                width=cross_width,
                label=attr,
                color=attr_colors[attr],
                edgecolor="#7a7a7a",
                linewidth=0.8,
                hatch="///",
                alpha=0.5,
                zorder=3
            )
        else:
            ax.bar(
                x + offsets[i],
                vals,
                width=regular_width,
                label=attr,
                color=attr_colors[attr],
                edgecolor="white",
                linewidth=0.7,
                zorder=2
            )

    for r_idx, idx in enumerate(panel_df.index):
        # best single
        row_single = panel_df.loc[idx, attr_order]
        best_single_attr = row_single.idxmax()
        best_single_val = row_single.max()

        i_single = plot_attr_order.index(best_single_attr)
        x_single = x[r_idx] + offsets[i_single]

        ax.text(
            x_single,
            best_single_val + ymax * 0.012,
            f"{best_single_val:.2f}",
            ha="center",
            va="bottom",
            fontsize=18,
            color="black"
        )

        # best cross
        cross_val = panel_df.loc[idx, "Best cross"]
        i_cross = plot_attr_order.index("Best cross")
        x_cross = x[r_idx] + offsets[i_cross]

        pair_name = best_cross_df.set_index("level_value").loc[idx, "pair_name"]
        pair_label = split_pair_name(pair_name)

        # pair name: move higher and darken slightly
        ax.text(
            x_cross,
            cross_val + ymax * 0.070,
            pair_label,
            ha="center",
            va="bottom",
            fontsize=13,
            color="#555555",
            linespacing=0.95
        )

        # cross value: same size as single-value label
        ax.text(
            x_cross,
            cross_val + ymax * 0.018,
            f"{cross_val:.2f}",
            ha="center",
            va="bottom",
            fontsize=18,
            color="black"
        )

    ax.set_title(title, pad=14)
    ax.set_ylabel("Between share")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.22, linewidth=0.7)
    ax.set_axisbelow(True)

    ax.legend(
        ncol=7,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.20),
        frameon=False,
        columnspacing=1.1,
        handlelength=1.6
    )

    fig.tight_layout(rect=[0, 0, 1, 0.92])

    plt.savefig(out_png, dpi=500, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

# =========================
# 5. region figure
# =========================
region_png = results_dir / "Figure2_region_between_share_with_best_cross_refined3.png"
region_pdf = results_dir / "Figure2_region_between_share_with_best_cross_refined3.pdf"

plot_panel(
    panel_df=region_panel,
    best_cross_df=region_best_cross,
    xlabels=region_labels,
    title="Region level between group share of total inequality",
    out_png=region_png,
    out_pdf=region_pdf
)

# =========================
# 6. country figure
# =========================
country_png = results_dir / "Figure3_selected_country_between_share_with_best_cross_refined3.png"
country_pdf = results_dir / "Figure3_selected_country_between_share_with_best_cross_refined3.pdf"

plot_panel(
    panel_df=country_panel,
    best_cross_df=country_best_cross,
    xlabels=country_labels,
    title="Selected country level between group share of total inequality",
    out_png=country_png,
    out_pdf=country_pdf
)

print(region_png)
print(region_pdf)
print(country_png)
print(country_pdf)
print(f"Unified y-axis max = {ymax:.3f}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import statsmodels.formula.api as smf

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"

# outputs
gdl_reg_data_out = process_dir / "gdl_regression_dataset.csv"
gdl_reg_long_out = process_dir / "gdl_regression_results_long.csv"
gdl_reg_wide_out = process_dir / "gdl_regression_results_wide.csv"

# =========================
# 1. read and prepare data
# =========================
df = pd.read_csv(input_path)

num_cols = [
    "EC2015_pc", "population",
    "SES_raw", "rural", "education", "age", "gender", "Hhsize"
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# keep usable rows
df = df[
    df["iso_code"].notna() &
    df["country"].notna() &
    df["analytical_region"].notna() &
    df["EC2015_pc"].notna() &
    df["population"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] >= 0)
].copy()

# dependent variable
df["ln_ecpc"] = np.log(df["EC2015_pc"] + 1)

# save prepared dataset
df.to_csv(gdl_reg_data_out, index=False)
print(f"Saved prepared dataset: {gdl_reg_data_out}")
print("Input shape for regression:", df.shape)

# =========================
# 2. model settings
# =========================
xvars = ["SES_raw", "rural", "education", "age", "gender", "Hhsize"]
selected_countries = ["Nigeria", "Egypt", "India", "China", "Germany", "Brazil", "United States"]

# global formulas
formula_global_multi_regionFE = "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize + C(analytical_region)"
formula_global_multi_countryFE = "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize + C(iso_code)"

# =========================
# 3. helper functions
# =========================
def stars(p):
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.1:
        return "*"
    return ""

def tidy_model(model, model_name, sample_name, formula_text):
    out = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "se": model.bse.values,
        "t": model.tvalues.values,
        "p": model.pvalues.values
    })
    out["stars"] = out["p"].apply(stars)
    out["coef_fmt"] = out["coef"].map(lambda x: f"{x:.4f}") + out["stars"]
    out["se_fmt"] = out["se"].map(lambda x: f"({x:.4f})")
    out["model"] = model_name
    out["sample"] = sample_name
    out["formula"] = formula_text
    out["nobs"] = int(model.nobs)
    out["r2"] = model.rsquared
    return out

def run_wls_cluster(df_in, formula, cluster_col, model_name, sample_name):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars + [cluster_col]).copy()
    if len(sub) == 0:
        return None, None

    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sub[cluster_col]}
    )
    tidy = tidy_model(model, model_name, sample_name, formula)
    return model, tidy

def run_wls_hc3(df_in, formula, model_name, sample_name):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars).copy()
    if len(sub) == 0:
        return None, None

    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(cov_type="HC3")
    tidy = tidy_model(model, model_name, sample_name, formula)
    return model, tidy

# =========================
# 4. run models
# =========================
all_tidy = []
model_summaries = []

tasks = []

# global models
tasks.append(("global_multi_regionFE", "Global", df.copy(), formula_global_multi_regionFE, "iso_code", "cluster"))
tasks.append(("global_multi_countryFE", "Global", df.copy(), formula_global_multi_countryFE, "iso_code", "cluster"))

# region-specific models
for region in sorted(df["analytical_region"].dropna().unique()):
    sub = df[df["analytical_region"] == region].copy()
    tasks.append((
        "region_multi",
        region,
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize",
        "iso_code",
        "cluster"
    ))

# selected-country models
for country in selected_countries:
    sub = df[df["country"] == country].copy()
    tasks.append((
        "country_multi",
        country,
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize",
        None,
        "hc3"
    ))

# single-variable global models
for x in xvars:
    tasks.append((
        f"global_single_{x}",
        "Global",
        df.copy(),
        f"ln_ecpc ~ {x} + C(analytical_region)",
        "iso_code",
        "cluster"
    ))

print(f"Total models to run: {len(tasks)}")

for model_name, sample_name, sub, formula, cluster_col, mode in tqdm(tasks, desc="Running regressions"):
    try:
        if mode == "cluster":
            model, tidy = run_wls_cluster(
                df_in=sub,
                formula=formula,
                cluster_col=cluster_col,
                model_name=model_name,
                sample_name=sample_name
            )
        else:
            model, tidy = run_wls_hc3(
                df_in=sub,
                formula=formula,
                model_name=model_name,
                sample_name=sample_name
            )

        if model is not None:
            all_tidy.append(tidy)
            model_summaries.append({
                "model": model_name,
                "sample": sample_name,
                "nobs": int(model.nobs),
                "r2": model.rsquared
            })
    except Exception as e:
        model_summaries.append({
            "model": model_name,
            "sample": sample_name,
            "nobs": np.nan,
            "r2": np.nan,
            "error": str(e)
        })

# =========================
# 5. save long results
# =========================
reg_long = pd.concat(all_tidy, ignore_index=True) if len(all_tidy) > 0 else pd.DataFrame()
reg_long.to_csv(gdl_reg_long_out, index=False)

summary_df = pd.DataFrame(model_summaries)

# =========================
# 6. save wide results
# =========================
# keep only core coefficients for easy reading
keep_terms = ["SES_raw", "rural", "education", "age", "gender", "Hhsize"]

wide_rows = []
main_models_for_wide = [
    "global_multi_regionFE",
    "global_multi_countryFE",
    "region_multi",
    "country_multi"
]

# global models
for term in keep_terms:
    row = {"group": "Global", "term": term}
    for model_name in ["global_multi_regionFE", "global_multi_countryFE"]:
        sub = reg_long[(reg_long["model"] == model_name) & (reg_long["term"] == term)]
        row[model_name] = sub["coef_fmt"].iloc[0] if len(sub) else ""
        row[f"{model_name}_se"] = sub["se_fmt"].iloc[0] if len(sub) else ""
    wide_rows.append(row)

# region models
for region in sorted(df["analytical_region"].dropna().unique()):
    for term in keep_terms:
        row = {"group": region, "term": term}
        sub = reg_long[
            (reg_long["model"] == "region_multi") &
            (reg_long["sample"] == region) &
            (reg_long["term"] == term)
        ]
        row["region_multi"] = sub["coef_fmt"].iloc[0] if len(sub) else ""
        row["region_multi_se"] = sub["se_fmt"].iloc[0] if len(sub) else ""
        wide_rows.append(row)

# selected-country models
for country in selected_countries:
    for term in keep_terms:
        row = {"group": country, "term": term}
        sub = reg_long[
            (reg_long["model"] == "country_multi") &
            (reg_long["sample"] == country) &
            (reg_long["term"] == term)
        ]
        row["country_multi"] = sub["coef_fmt"].iloc[0] if len(sub) else ""
        row["country_multi_se"] = sub["se_fmt"].iloc[0] if len(sub) else ""
        wide_rows.append(row)

# single-variable global models
for x in xvars:
    row = {"group": "Global_single", "term": x}
    sub = reg_long[(reg_long["model"] == f"global_single_{x}") & (reg_long["term"] == x)]
    row[f"global_single_{x}"] = sub["coef_fmt"].iloc[0] if len(sub) else ""
    row[f"global_single_{x}_se"] = sub["se_fmt"].iloc[0] if len(sub) else ""
    wide_rows.append(row)

# add model stats block
for _, srow in summary_df.iterrows():
    wide_rows.append({
        "group": srow.get("sample", ""),
        "term": f"N | {srow.get('model', '')}",
        "value": srow.get("nobs", "")
    })
    wide_rows.append({
        "group": srow.get("sample", ""),
        "term": f"R2 | {srow.get('model', '')}",
        "value": srow.get("r2", "")
    })

reg_wide = pd.DataFrame(wide_rows)
reg_wide.to_csv(gdl_reg_wide_out, index=False)

# =========================
# 7. print summary
# =========================
print("\nSaved files:")
print(gdl_reg_long_out)
print(gdl_reg_wide_out)

print("\nModel summaries:")
print(summary_df.sort_values(["model", "sample"]).to_string(index=False))

print("\nGlobal main models:")
global_summary = summary_df[summary_df["model"].isin(["global_multi_regionFE", "global_multi_countryFE"])]
print(global_summary.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import statsmodels.formula.api as smf

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"

region_heatmap_png = results_dir / "GDL_region_regression_heatmap_coolwarm_greycenter.png"
region_heatmap_pdf = results_dir / "GDL_region_regression_heatmap_coolwarm_greycenter.pdf"

country_heatmap_png = results_dir / "GDL_selected_country_regression_heatmap_coolwarm_greycenter.png"
country_heatmap_pdf = results_dir / "GDL_selected_country_regression_heatmap_coolwarm_greycenter.pdf"

global_coef_png = results_dir / "GDL_global_countryFE_coefficient_plot_final.png"
global_coef_pdf = results_dir / "GDL_global_countryFE_coefficient_plot_final.pdf"

# =========================
# 1. read data
# =========================
df = pd.read_csv(input_path)

num_cols = [
    "EC2015_pc", "population",
    "SES_raw", "rural", "education", "age", "gender", "Hhsize"
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["iso_code"].notna() &
    df["country"].notna() &
    df["analytical_region"].notna() &
    df["EC2015_pc"].notna() &
    df["population"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] >= 0)
].copy()

df["ln_ecpc"] = np.log(df["EC2015_pc"] + 1)

xvars = ["SES_raw", "rural", "education", "age", "gender", "Hhsize"]
selected_countries = ["Nigeria", "Egypt", "India", "China", "Germany", "Brazil", "United States"]

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

country_order = ["Nigeria", "Egypt", "India", "China", "Germany", "Brazil", "United States"]

region_labels = {
    "Sub-Saharan Africa": "Sub-Saharan\nAfrica",
    "Northern Africa and Western Asia": "Northern Africa\nand Western Asia",
    "South Asia": "South Asia",
    "Eastern and South-eastern Asia": "Eastern and\nSouth-eastern Asia",
    "Europe": "Europe",
    "Latin America and the Caribbean": "Latin America\nand the Caribbean",
    "Northern America and Oceania": "Northern America\nand Oceania"
}

display_term_labels = {
    "SES_raw": "SES",
    "rural": "Rural",
    "education": "Education",
    "age": "Age",
    "gender": "Gender",
    "Hhsize": "Hhsize"
}

# =========================
# 2. helper
# =========================
def stars(p):
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.1:
        return "*"
    return ""

def run_wls_cluster(df_in, formula, cluster_col):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars + [cluster_col]).copy()
    if len(sub) == 0:
        return None
    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sub[cluster_col]}
    )
    return model

def run_wls_hc3(df_in, formula):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars).copy()
    if len(sub) == 0:
        return None
    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(cov_type="HC3")
    return model

def extract_core_terms(model, sample_name):
    rows = []
    for v in xvars:
        if v in model.params.index:
            rows.append({
                "sample": sample_name,
                "term": v,
                "coef": model.params[v],
                "se": model.bse[v],
                "p": model.pvalues[v],
                "stars": stars(model.pvalues[v])
            })
    return pd.DataFrame(rows)

def build_heatmap_df(res_df, sample_order):
    coef_mat = (
        res_df.pivot(index="sample", columns="term", values="coef")
        .reindex(sample_order)[xvars]
    )
    star_mat = (
        res_df.pivot(index="sample", columns="term", values="stars")
        .reindex(sample_order)[xvars]
    )
    return coef_mat, star_mat

def make_grey_centered_coolwarm():
    base = plt.cm.get_cmap("coolwarm", 256)
    colors = base(np.linspace(0, 1, 256))
    grey = np.array(mcolors.to_rgba("#d9d9d9"))
    colors[124:132] = grey
    return mcolors.ListedColormap(colors)

# =========================
# 3. run region models
# =========================
region_rows = []
for region in region_order:
    sub = df[df["analytical_region"] == region].copy()
    model = run_wls_cluster(
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize",
        "iso_code"
    )
    if model is not None:
        region_rows.append(extract_core_terms(model, region))
region_res = pd.concat(region_rows, ignore_index=True)

# =========================
# 4. run selected country models
# =========================
country_rows = []
for country in country_order:
    sub = df[df["country"] == country].copy()
    model = run_wls_hc3(
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize"
    )
    if model is not None:
        country_rows.append(extract_core_terms(model, country))
country_res = pd.concat(country_rows, ignore_index=True)

# =========================
# 5. run global country FE model
# =========================
global_model = run_wls_cluster(
    df,
    "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize + C(iso_code)",
    "iso_code"
)

global_res = extract_core_terms(global_model, "Global_countryFE")
global_res["ci_low"] = global_res["coef"] - 1.96 * global_res["se"]
global_res["ci_high"] = global_res["coef"] + 1.96 * global_res["se"]

# =========================
# 6. heatmap prep
# =========================
region_coef, region_star = build_heatmap_df(region_res, region_order)
country_coef, country_star = build_heatmap_df(country_res, country_order)

absmax = np.nanmax([
    np.abs(region_coef.values).max(),
    np.abs(country_coef.values).max()
])
absmax = max(absmax, 0.01)

coef_df = global_res.copy()
coef_df["term_label"] = coef_df["term"].map(display_term_labels)
coef_df = coef_df.set_index("term").loc[xvars].reset_index()

# =========================
# 7. plotting style
# =========================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12

heatmap_cmap = make_grey_centered_coolwarm()

# =========================
# 8. region heatmap
# =========================
fig, ax = plt.subplots(figsize=(9, 6.8))

im = ax.imshow(
    region_coef.values,
    cmap=heatmap_cmap,
    vmin=-absmax,
    vmax=absmax,
    aspect="auto"
)

ax.set_xticks(np.arange(len(xvars)))
ax.set_xticklabels([display_term_labels[v] for v in xvars])
ax.set_yticks(np.arange(len(region_order)))
ax.set_yticklabels([region_labels[r] for r in region_order])

for i in range(region_coef.shape[0]):
    for j in range(region_coef.shape[1]):
        val = region_coef.iloc[i, j]
        st = region_star.iloc[i, j]
        if pd.notna(val):
            txt = f"{val:.2f}{st}"
            color = "black" if abs(val) < absmax * 0.55 else "white"
            ax.text(j, i, txt, ha="center", va="center", fontsize=10, color=color)

ax.set_title("Regional regression coefficients")
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cbar.set_label("Coefficient")

fig.tight_layout()
plt.savefig(region_heatmap_png, dpi=500, bbox_inches="tight")
plt.savefig(region_heatmap_pdf, bbox_inches="tight")
plt.close(fig)

# =========================
# 9. country heatmap
# =========================
fig, ax = plt.subplots(figsize=(9, 6.4))

im = ax.imshow(
    country_coef.values,
    cmap=heatmap_cmap,
    vmin=-absmax,
    vmax=absmax,
    aspect="auto"
)

ax.set_xticks(np.arange(len(xvars)))
ax.set_xticklabels([display_term_labels[v] for v in xvars])
ax.set_yticks(np.arange(len(country_order)))
ax.set_yticklabels(country_order)

for i in range(country_coef.shape[0]):
    for j in range(country_coef.shape[1]):
        val = country_coef.iloc[i, j]
        st = country_star.iloc[i, j]
        if pd.notna(val):
            txt = f"{val:.2f}{st}"
            color = "black" if abs(val) < absmax * 0.55 else "white"
            ax.text(j, i, txt, ha="center", va="center", fontsize=10, color=color)

ax.set_title("Selected country regression coefficients")
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cbar.set_label("Coefficient")

fig.tight_layout()
plt.savefig(country_heatmap_png, dpi=500, bbox_inches="tight")
plt.savefig(country_heatmap_pdf, bbox_inches="tight")
plt.close(fig)

# =========================
# 10. global coefficient plot
# =========================
fig, ax = plt.subplots(figsize=(7.8, 4.8))

y = np.arange(len(coef_df))
ax.axvline(0, color="#888888", linewidth=1.0, linestyle="--")

ax.errorbar(
    coef_df["coef"],
    y,
    xerr=1.96 * coef_df["se"],
    fmt="o",
    color="#7b3294",
    ecolor="#7b3294",
    elinewidth=1.4,
    capsize=3,
    markersize=6
)

for i, row in coef_df.iterrows():
    label = f"{row['coef']:.2f}{row['stars']}"
    x_text = row["ci_high"] + 0.01 if row["coef"] >= 0 else row["ci_low"] - 0.01
    ha = "left" if row["coef"] >= 0 else "right"
    ax.text(x_text, i, label, va="center", ha=ha, fontsize=11)

ax.set_yticks(y)
ax.set_yticklabels(coef_df["term_label"])
ax.invert_yaxis()
ax.set_xlabel("Coefficient with 95% CI")
ax.set_title("Global model with country fixed effects")
ax.grid(axis="x", alpha=0.2, linewidth=0.6)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(global_coef_png, dpi=500, bbox_inches="tight")
plt.savefig(global_coef_pdf, bbox_inches="tight")
plt.close(fig)

print(region_heatmap_png)
print(region_heatmap_pdf)
print(country_heatmap_png)
print(country_heatmap_pdf)
print(global_coef_png)
print(global_coef_pdf)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# =========================
# 0. paths
# =========================
base_dir = Path("/Users/nico/Documents/PhD research/energy poverty")
process_dir = base_dir / "process"
results_dir = base_dir / "results"

process_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

input_path = process_dir / "df2015_analytical.csv"

regional_png = results_dir / "regional_small_multiples_coefplot_ordered_colored_v2.png"
country_png = results_dir / "selected_country_small_multiples_coefplot_ordered_colored_v2.png"

# =========================
# 1. read data
# =========================
df = pd.read_csv(input_path)

num_cols = [
    "EC2015_pc", "population",
    "SES_raw", "rural", "education", "age", "gender", "Hhsize"
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["iso_code"].notna() &
    df["country"].notna() &
    df["analytical_region"].notna() &
    df["EC2015_pc"].notna() &
    df["population"].notna() &
    (df["population"] > 0) &
    (df["EC2015_pc"] >= 0)
].copy()

df["ln_ecpc"] = np.log(df["EC2015_pc"] + 1)

xvars = ["SES_raw", "rural", "education", "age", "gender", "Hhsize"]
term_labels = {
    "SES_raw": "SES",
    "rural": "Rural",
    "education": "Education",
    "age": "Age",
    "gender": "Gender",
    "Hhsize": "Hhsize"
}

# order exactly as your figure
region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "Eastern and South-eastern Asia",
    "Latin America and the Caribbean",
    "South Asia",
    "Europe",
    "Northern America and Oceania"
]

country_order = [
    "Nigeria",
    "Egypt",
    "China",
    "Brazil",
    "India",
    "Germany",
    "United States"
]

# titles
region_titles = {
    "Sub-Saharan Africa": "Sub-Saharan Africa",
    "Northern Africa and Western Asia": "Northern Africa and Western Asia",
    "Eastern and South-eastern Asia": "Eastern and South-eastern Asia",
    "Latin America and the Caribbean": "Latin America and the Caribbean",
    "South Asia": "South Asia",
    "Europe": "Europe",
    "Northern America and Oceania": "Northern America and Oceania"
}

# colors matched to your Lorenz/Gini palette style
region_colors = {
    "Sub-Saharan Africa": "#9b2d2b",
    "Northern Africa and Western Asia": "#c76e33",
    "Eastern and South-eastern Asia": "#2f6f95",
    "Latin America and the Caribbean": "#7b5aac",
    "South Asia": "#d39b0a",
    "Europe": "#4b9150",
    "Northern America and Oceania": "#c05a8f"
}

country_colors = {
    "Nigeria": "#9b2d2b",
    "Egypt": "#e6865a",
    "China": "#2b6cb0",
    "Brazil": "#7a2e8a",
    "India": "#d89c00",
    "Germany": "#1b7f3c",
    "United States": "#cc2b7f"
}

# =========================
# 2. helpers
# =========================
def stars(p):
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.1:
        return "*"
    return ""

def run_wls_cluster(df_in, formula, cluster_col):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars + [cluster_col]).copy()
    if len(sub) == 0:
        return None
    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sub[cluster_col]}
    )
    return model

def run_wls_hc3(df_in, formula):
    sub = df_in.dropna(subset=["ln_ecpc", "population"] + xvars).copy()
    if len(sub) == 0:
        return None
    model = smf.wls(
        formula=formula,
        data=sub,
        weights=sub["population"]
    ).fit(cov_type="HC3")
    return model

def extract_terms(model, sample_name):
    rows = []
    for v in xvars:
        if v in model.params.index:
            coef = model.params[v]
            se = model.bse[v]
            p = model.pvalues[v]
            rows.append({
                "sample": sample_name,
                "term": v,
                "coef": coef,
                "se": se,
                "p": p,
                "stars": stars(p),
                "ci_low": coef - 1.96 * se,
                "ci_high": coef + 1.96 * se
            })
    return pd.DataFrame(rows)

# =========================
# 3. run regional models
# =========================
region_rows = []
for region in region_order:
    sub = df[df["analytical_region"] == region].copy()
    model = run_wls_cluster(
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize",
        "iso_code"
    )
    if model is not None:
        region_rows.append(extract_terms(model, region))

region_res = pd.concat(region_rows, ignore_index=True)

# =========================
# 4. run selected-country models
# =========================
country_rows = []
for country in country_order:
    sub = df[df["country"] == country].copy()
    model = run_wls_hc3(
        sub,
        "ln_ecpc ~ SES_raw + rural + education + age + gender + Hhsize"
    )
    if model is not None:
        country_rows.append(extract_terms(model, country))

country_res = pd.concat(country_rows, ignore_index=True)

# =========================
# 5. plotting style
# =========================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 15

all_bounds = np.concatenate([
    region_res["ci_low"].values,
    region_res["ci_high"].values,
    country_res["ci_low"].values,
    country_res["ci_high"].values
])
xmax = np.nanmax(np.abs(all_bounds)) * 1.18
text_offset = xmax * 0.03

# =========================
# 6. regional small multiples
# layout:
# top: SSA, NAWA, ESEA, LAC
# bottom: South Asia, Europe, Northern America and Oceania
# =========================
fig, axes = plt.subplots(2, 4, figsize=(16, 8.8), sharex=True, sharey=True)
axes = axes.flatten()

for i, region in enumerate(region_order):
    ax = axes[i]
    sub = region_res[region_res["sample"] == region].copy()
    sub["term_label"] = sub["term"].map(term_labels)
    sub = sub.set_index("term").loc[xvars].reset_index()

    y = np.arange(len(sub))
    ax.axvline(0, color="#999999", linestyle="--", linewidth=1)

    this_color = region_colors[region]

    ax.errorbar(
        sub["coef"], y,
        xerr=1.96 * sub["se"],
        fmt="o",
        color=this_color,
        ecolor=this_color,
        elinewidth=1.4,
        capsize=2.8,
        markersize=5.8
    )

    for j, row in sub.iterrows():
        label = f"{row['coef']:.2f}{row['stars']}"
        x_text = row["ci_high"] + text_offset if row["coef"] >= 0 else row["ci_low"] - text_offset
        ha = "left" if row["coef"] >= 0 else "right"
        ax.text(x_text, j, label, va="center", ha=ha, fontsize=12)

    ax.set_title(region_titles[region], pad=10, fontsize=16)
    ax.set_xlim(-xmax, xmax)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["term_label"], fontsize=16)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.15)

# last empty panel
axes[-1].axis("off")

fig.suptitle("Regional regression coefficients", fontsize=22, y=0.97)
fig.tight_layout(rect=[0, 0, 1, 0.94])

plt.savefig(regional_png, dpi=500, bbox_inches="tight")
plt.close(fig)

# =========================
# 7. selected-country small multiples
# layout:
# top: Nigeria, Egypt, China, Brazil
# bottom: India, Germany, United States
# =========================
fig, axes = plt.subplots(2, 4, figsize=(16, 8.8), sharex=True, sharey=True)
axes = axes.flatten()

for i, country in enumerate(country_order):
    ax = axes[i]
    sub = country_res[country_res["sample"] == country].copy()
    sub["term_label"] = sub["term"].map(term_labels)
    sub = sub.set_index("term").loc[xvars].reset_index()

    y = np.arange(len(sub))
    ax.axvline(0, color="#999999", linestyle="--", linewidth=1)

    this_color = country_colors[country]

    ax.errorbar(
        sub["coef"], y,
        xerr=1.96 * sub["se"],
        fmt="o",
        color=this_color,
        ecolor=this_color,
        elinewidth=1.4,
        capsize=2.8,
        markersize=5.8
    )

    for j, row in sub.iterrows():
        label = f"{row['coef']:.2f}{row['stars']}"
        x_text = row["ci_high"] + text_offset if row["coef"] >= 0 else row["ci_low"] - text_offset
        ha = "left" if row["coef"] >= 0 else "right"
        ax.text(x_text, j, label, va="center", ha=ha, fontsize=12)

    ax.set_title(country, pad=10, fontsize=16)
    ax.set_xlim(-xmax, xmax)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["term_label"], fontsize=16)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.15)

# last empty panel
axes[-1].axis("off")

fig.suptitle("Selected country regression coefficients", fontsize=22, y=0.97)
fig.tight_layout(rect=[0, 0, 1, 0.94])

plt.savefig(country_png, dpi=500, bbox_inches="tight")
plt.close(fig)

print(regional_png)
print(country_png)